# EMA Cluster Setup — Research & Signal Engine (NIFTY 200)

**Separate system.** Independent from any Bollinger Band / Heikin-Ashi
notebook; nothing here modifies that system.

## Source concept (as given)
```
EMA 10, EMA 20, EMA 89, EMA 200
1. EMA cluster / crossover setup
2. Check on DAILY timeframe first
3. Validate on WEEKLY timeframe
4. EMA200 = major structural trend filter
5. EMA10/20/89 = shorter/intermediate trend alignment
6. After the cluster condition develops, look for a continuation/trend
   entry using swing structure
7. Stop loss = the relevant swing low (longs)
8. Reverse logic applies for shorts
```

## How ambiguity is handled
The handwritten note does not give one precise mathematical definition of
an "EMA cluster crossover," nor an exact entry timing after it. **No single
interpretation is hard-coded.** Every rule below is labeled:

| Label | Meaning |
|---|---|
| **SOURCE-DERIVED RULE** | Directly stated in the handwritten note |
| **MATHEMATICAL DEFINITION** | An exact, reproducible formula implementing a source-derived rule |
| **RESEARCH HYPOTHESIS** | A specific, testable interpretation of an ambiguous part of the note — not assumed true |
| **OPTIONAL FILTER** | A condition the user can enable/disable; never silently mandatory |
| **EMPIRICALLY TESTED RESULT** | An output produced by running the above against price history — a finding, not an assumption |

Four independent cluster-crossover **Definitions A–D** and five independent
**Entry Models A–E** are implemented and reported *separately*. None is
presented as "the" correct interpretation — Section 30's instruction not to
call 10/20/89/200 "optimal" is followed throughout.

## Three modes (kept separate, never mixed)
- **Mode 1 — Live Scan**: what setups exist right now?
- **Mode 2 — Historical Event Study**: what happened after historical setups?
- **Mode 3 — Walk-Forward Research**: does it survive honest out-of-sample testing?

## Data-access note
Live-data cells need network access to `nseindia.com` and Yahoo Finance.
If that access is unavailable, the notebook **loudly** falls back to a
synthetic demo universe (every value tagged
`SYNTHETIC_DEMO_DATA_NOT_REAL_MARKET_DATA`) purely to exercise the plumbing
top-to-bottom. Never treat that fallback's numbers as a real finding.


## Cell 1 — Install

In [ ]:
!pip install -q yfinance openpyxl requests pyarrow


## Cell 2 — Imports

`yfinance` is imported lazily inside the data-layer function (Cell 5), so this cell does not hard-fail if the install step is still propagating.

In [ ]:
import io, os, time, json, hashlib, warnings, itertools
import datetime as dt

import numpy as np
import pandas as pd
import requests

from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports OK.")


## Cell 3 — Configuration (Section 44: no magic numbers)

Every parameter that changes the strategy lives here, visibly, with a note on whether it is a **SOURCE-DERIVED RULE** (the user's own spec) or a **RESEARCH HYPOTHESIS** default that must be empirically validated, not assumed.

In [ ]:
class Config:
    # --- EMA Cluster core periods -- SOURCE-DERIVED RULE (user's own spec) ---
    EMA_FAST = 10
    EMA_MEDIUM = 20
    EMA_STRUCTURAL = 89
    EMA_LONG = 200

    # --- EMA seeding method -- MATHEMATICAL DEFINITION, explicit ---
    # 'sma'       : seed EMA at bar (period-1) with the simple mean of the
    #               first `period` observations, then recurse (textbook /
    #               TA-Lib convention). This is the DEFAULT.
    # 'first_obs' : seed EMA_0 = Close_0, recurse from bar 0 (naive
    #               alternative, documented but not default).
    EMA_SEED_METHOD = "sma"

    # --- Swing structure ---
    SWING_LEFT_BARS = 5
    SWING_RIGHT_BARS = 5   # confirmation lag: a swing is only known this many bars later

    # --- Volatility ---
    ATR_PERIOD = 14

    # --- Cluster compression / expansion -- RESEARCH HYPOTHESIS defaults ---
    CLUSTER_WIDTH_LOOKBACK = 252          # rolling window for percentile ranking
    CLUSTER_COMPRESSION_PCTL = 0.20       # must be tested, not assumed
    CLUSTER_EXPANSION_PCTL = 0.80

    # --- Sequential crossover window (Definition A) -- RESEARCH HYPOTHESIS ---
    SEQUENTIAL_CROSSOVER_MAX_GAP = 10     # "X sessions" in the handwritten note

    # --- Pullback classification bands (ATR multiples) -- RESEARCH HYPOTHESIS ---
    PULLBACK_SHALLOW_ATR = 1.0
    PULLBACK_MODERATE_ATR = 2.5
    PULLBACK_DEEP_ATR = 4.0

    # --- Data / warm-up ---
    MIN_HISTORY_YEARS = 4
    MAX_STALE_DAYS = 5

    # --- Risk engine -- OPTIONAL / user-set ---
    RISK_PER_TRADE_PCT = 0.5
    ATR_STOP_MULTIPLE = 2.5   # kept separate from the swing-based stop

    # --- Execution assumption (Section 18) ---
    EXECUTION_MODEL = "next_open"   # 'next_open' | 'next_close' | 'same_close'

    # --- Transaction costs (Section 31) -- placeholders, user must confirm ---
    BROKERAGE_PCT = 0.03
    STT_PCT = 0.10
    EXCHANGE_CHARGES_PCT = 0.00345
    GST_PCT = 0.18
    SLIPPAGE_PCT = 0.05

    # --- Liquidity (Section 22) -- OFF by default, must be explicitly enabled ---
    ENABLE_LIQUIDITY_FILTER = False
    MIN_AVG_DOLLAR_TURNOVER = 0.0

    # --- Market regime EMAs (index-level) ---
    INDEX_EMA_FAST = 20
    INDEX_EMA_MEDIUM = 50
    INDEX_EMA_STRUCTURAL = 89
    INDEX_EMA_LONG = 200

    RANDOM_SEED = 42


CFG = Config()
print("Config loaded. EMA periods:", CFG.EMA_FAST, CFG.EMA_MEDIUM, CFG.EMA_STRUCTURAL, CFG.EMA_LONG)


## Cell 4 — Universe: NIFTY 200 constituents (Section 41)

**SOURCE-DERIVED RULE**: the universe is NIFTY 200. **The list is never hard-coded.** If the official NSE source cannot be fetched, the scan **fails loudly** rather than silently falling back to a stale/partial list.

In [ ]:
SYNTHETIC_DEMO_TAG = "SYNTHETIC_DEMO_DATA_NOT_REAL_MARKET_DATA"
NIFTY200_CSV_URL = "https://archives.nseindia.com/content/indices/ind_nifty200list.csv"


class UniverseFetchError(RuntimeError):
    """Raised when the official NIFTY 200 list cannot be retrieved. Callers
    must NOT catch this and substitute a hard-coded list (Section 41)."""
    pass


def fetch_nifty200_universe(timeout: int = 15) -> pd.DataFrame:
    """
    Downloads the CURRENT official NIFTY 200 constituent list from NSE.
    NSE's archive endpoints commonly reject bare requests (403) unless a
    browser-like session is established first, so we warm a session against
    the public NSE homepage before requesting the CSV. This is standard
    practice for NSE's public index-constituent files, not a bypass of any
    access control on non-public data.
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
        "Accept": "text/csv,application/csv,*/*",
        "Referer": "https://www.nseindia.com/market-data/live-equity-market",
    }
    try:
        session = requests.Session()
        session.headers.update(headers)
        session.get("https://www.nseindia.com", timeout=timeout)  # warm cookies
        resp = session.get(NIFTY200_CSV_URL, timeout=timeout)
        resp.raise_for_status()
        df = pd.read_csv(io.StringIO(resp.text))
    except Exception as e:
        raise UniverseFetchError(
            f"Failed to fetch official NIFTY 200 list from NSE: {e}. Per the "
            f"research design, the scan is ABORTED rather than falling back "
            f"to a stale/hard-coded universe."
        ) from e

    if "Symbol" not in df.columns or len(df) < 150:
        raise UniverseFetchError(
            f"NIFTY 200 list fetched but failed a sanity check "
            f"(columns={list(df.columns)}, rows={len(df)}). Aborting rather "
            f"than proceeding on a suspect universe."
        )
    df["YF_Symbol"] = df["Symbol"].astype(str).str.strip() + ".NS"
    return df


def label_universe_mode(historical_membership_available: bool) -> str:
    """Section 27/41: explicit labeling to avoid silent survivorship bias."""
    if historical_membership_available:
        return "HISTORICAL-MEMBERSHIP-BACKTEST"
    return "CURRENT-UNIVERSE HISTORICAL SIMULATION"


print("Universe layer defined. This scan will use:",
      label_universe_mode(historical_membership_available=False),
      "(no historical NIFTY-200 constituent history is wired up here — see Section 27 note).")


## Cell 5 — Data layer: yfinance + local caching (Sections 42/43)

In [ ]:
CACHE_DIR = "./ema_cluster_cache"


def _cache_path(key: str) -> str:
    os.makedirs(CACHE_DIR, exist_ok=True)
    h = hashlib.sha256(key.encode()).hexdigest()[:16]
    return os.path.join(CACHE_DIR, f"{h}.parquet")


def get_daily_ohlc(symbol: str, start: str, end: str = None, use_cache: bool = True,
                    max_retries: int = 3, retry_sleep: float = 2.0) -> pd.DataFrame:
    """Thin, swappable data-provider wrapper. Caches to local parquet keyed by
    (symbol, start, end) so repeated runs avoid redundant network calls.
    Retries transient failures; raises after exhausting retries rather than
    silently returning empty/partial data."""
    import yfinance as yf  # lazy import: keeps Cell 2 import-safe if install lags
    end = end or dt.date.today().isoformat()
    cache_key = f"yfinance|{symbol}|{start}|{end}"
    path = _cache_path(cache_key)
    if use_cache and os.path.exists(path):
        return pd.read_parquet(path)

    last_err = None
    for attempt in range(max_retries):
        try:
            df = yf.download(symbol, start=start, end=end, progress=False, auto_adjust=False)
            if df is None or df.empty:
                raise ValueError(f"No data returned for {symbol}")
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df.rename(columns={"Adj Close": "AdjClose"})
            df.index.name = "Date"
            if use_cache:
                df.to_parquet(path)
            return df
        except Exception as e:
            last_err = e
            time.sleep(retry_sleep * (attempt + 1))
    raise RuntimeError(f"Failed to download {symbol} after {max_retries} attempts: {last_err}")


def batch_download(symbols: list, start: str, end: str = None, use_cache: bool = True) -> dict:
    """{symbol: df_or_None}. Failures are caught PER SYMBOL by the caller via
    ScanErrorLog (Cell 6) - this function does not swallow errors silently."""
    out = {}
    for sym in symbols:
        try:
            out[sym] = get_daily_ohlc(sym, start=start, end=end, use_cache=use_cache)
        except Exception as e:
            out[sym] = None
    return out


print("Data layer defined (yfinance + parquet cache at", CACHE_DIR, ").")


## Cell 6 — Data validation (Section 36) & error handling (Section 35)

A single broken ticker must never halt the full scan; every failure captures ticker/stage/error_type/error_message/timestamp — never a bare `except: pass`.

In [ ]:
class ScanErrorLog:
    def __init__(self):
        self.rows = []

    def log(self, ticker: str, stage: str, error: Exception):
        self.rows.append({
            "ticker": ticker, "stage": stage,
            "error_type": type(error).__name__, "error_message": str(error),
            "timestamp": dt.datetime.utcnow().isoformat(),
        })

    def to_df(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows, columns=["ticker", "stage", "error_type", "error_message", "timestamp"])


def run_stage_safely(ticker: str, stage_name: str, fn, error_log: ScanErrorLog, *args, **kwargs):
    """Wraps one pipeline stage for one ticker. On failure, logs full context
    and returns None so the caller can skip this ticker WITHOUT halting the scan."""
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        error_log.log(ticker, stage_name, e)
        return None


def validate_ohlc(df: pd.DataFrame, ticker: str, max_stale_days: int = 5,
                   min_history_days: int = 260) -> dict:
    issues = []
    if df is None or df.empty:
        return {"ticker": ticker, "passed": False, "issues": ["empty_or_missing_dataframe"]}

    required_cols = ["Open", "High", "Low", "Close", "Volume"]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        issues.append(f"missing_columns:{missing_cols}")

    if df.index.duplicated().any():
        issues.append("duplicate_timestamps")
    if not df.index.is_monotonic_increasing:
        issues.append("non_monotonic_dates")
    if len(df) < min_history_days:
        issues.append(f"insufficient_history:{len(df)}_rows")

    numeric_cols = [c for c in required_cols if c in df.columns and c != "Volume"]
    if numeric_cols:
        if (df[numeric_cols] <= 0).any().any():
            issues.append("zero_or_negative_prices")
        if df[numeric_cols].isna().any().any():
            issues.append("missing_ohlc_values")
        if "High" in df.columns and "Low" in df.columns:
            bad_hl = (df["High"] < df[["Open", "Close", "Low"]].max(axis=1)) | \
                     (df["Low"] > df[["Open", "Close", "High"]].min(axis=1))
            if bad_hl.any():
                issues.append(f"inconsistent_high_low_bars:{int(bad_hl.sum())}")

    if len(df) > 0:
        last_date = df.index.max()
        now = pd.Timestamp.now(tz=last_date.tz) if last_date.tz else pd.Timestamp.now()
        stale_days = (now - last_date).days
        if stale_days > max_stale_days:
            issues.append(f"stale_data:{stale_days}_days_old")

    if "Close" in df.columns and len(df) > 5:
        ret = df["Close"].pct_change().abs()
        abnormal_gaps = ret[ret > 0.35]
        if len(abnormal_gaps) > 0:
            issues.append(f"abnormal_gaps_flagged_for_review:{len(abnormal_gaps)}")

    hard_issues = [i for i in issues if not i.startswith("abnormal_gaps")]
    return {"ticker": ticker, "passed": len(hard_issues) == 0, "issues": issues}


print("Data validation + error handling defined.")


## Cell 7 — Daily EMA engine (Sections 2–3)

**MATHEMATICAL DEFINITION**: `EMA_t = (Close_t - EMA_{t-1}) * mult + EMA_{t-1}`, `mult = 2/(N+1)`. The recursion is implemented explicitly (not via a library default) so the **seeding choice is visible** — see Cell 22 for the cross-check against a trusted reference implementation.

In [ ]:
def compute_ema(series: pd.Series, period: int, seed_method: str = None) -> pd.Series:
    """
    seed_method='sma'       : EMA is NaN for the first period-1 bars; EMA at
                               bar (period-1) is seeded as the simple mean of
                               the first `period` observations (textbook /
                               TA-Lib convention). DEFAULT.
    seed_method='first_obs' : EMA_0 = Close_0 (naive alternative, documented,
                               not the default).
    """
    if seed_method is None:
        seed_method = CFG.EMA_SEED_METHOD
    values = series.to_numpy(dtype=float)
    n = len(values)
    out = np.full(n, np.nan)
    multiplier = 2.0 / (period + 1.0)

    if seed_method == "sma":
        if n < period:
            return pd.Series(out, index=series.index)
        seed_idx = period - 1
        out[seed_idx] = np.nanmean(values[:period])
        for t in range(seed_idx + 1, n):
            out[t] = (values[t] - out[t - 1]) * multiplier + out[t - 1]
    elif seed_method == "first_obs":
        if n == 0:
            return pd.Series(out, index=series.index)
        out[0] = values[0]
        for t in range(1, n):
            out[t] = (values[t] - out[t - 1]) * multiplier + out[t - 1]
    else:
        raise ValueError(f"Unknown EMA seed_method: {seed_method}")
    return pd.Series(out, index=series.index)


def compute_all_emas(df: pd.DataFrame, price_col: str = "Close", suffix: str = "", cfg: Config = None) -> pd.DataFrame:
    """Attach EMA10/20/89/200 (role-named columns) to a copy of df.
    `cfg` lets the Sensitivity cell (21) perturb periods without mutating
    the shared global CFG."""
    if cfg is None:
        cfg = CFG
    out = df.copy()
    for period, name in [(cfg.EMA_FAST, "EMA10"), (cfg.EMA_MEDIUM, "EMA20"),
                          (cfg.EMA_STRUCTURAL, "EMA89"), (cfg.EMA_LONG, "EMA200")]:
        out[f"{name}{suffix}"] = compute_ema(out[price_col], period, seed_method=cfg.EMA_SEED_METHOD)
    return out


print("Daily EMA engine defined.")


## Cell 8 — Weekly EMA engine: true weekly OHLC, zero leakage (Section 4)

**SOURCE-DERIVED RULE**: weekly = structural context, daily = setup/entry context. **CRITICAL**: a daily signal on day T may only use the latest weekly candle that had *already fully closed* by day T.

In [ ]:
def build_true_weekly_ohlc(daily: pd.DataFrame) -> pd.DataFrame:
    """Resamples proper weekly OHLC (Mon-Fri trading week, W-FRI anchor) from
    daily bars - never daily EMAs mislabeled as weekly."""
    daily = daily.sort_index()
    weekly = daily.resample("W-FRI").agg(
        {"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"}
    )
    weekly = weekly.dropna(subset=["Open", "High", "Low", "Close"])
    return weekly


def attach_last_known_weekly(daily_with_emas: pd.DataFrame, weekly_with_emas: pd.DataFrame,
                              weekly_cols) -> pd.DataFrame:
    """As-of merges each daily row with the latest weekly row whose week-end
    label is STRICTLY BEFORE that daily date (never same-day or future)."""
    d = daily_with_emas.sort_index().copy()
    w = weekly_with_emas.sort_index().copy()[weekly_cols]
    d["_asof_key"] = (d.index - pd.Timedelta(nanoseconds=1)).astype("datetime64[ns]")
    left = d.reset_index().rename(columns={"index": "Date"}).sort_values("_asof_key")
    right = w.reset_index().rename(columns={"index": "WeekEnd"})
    right["WeekEnd"] = right["WeekEnd"].astype("datetime64[ns]")
    right = right.sort_values("WeekEnd")
    merged = pd.merge_asof(left, right, left_on="_asof_key", right_on="WeekEnd", direction="backward")
    merged = merged.set_index("Date").drop(columns=["_asof_key"])
    return merged


def latest_known_weekly_row(weekly_with_emas: pd.DataFrame, as_of_date: pd.Timestamp) -> pd.Series:
    """Scalar version of the no-lookahead rule above, used in the validation tests (Cell 22)."""
    eligible = weekly_with_emas[weekly_with_emas.index < as_of_date]
    return None if eligible.empty else eligible.iloc[-1]


print("Weekly EMA engine defined (no-lookahead as-of join).")


## Cell 9 — EMA state & crossover engine (Sections 5–6, 12)

*Consolidated cell*: EMA alignment state, pairwise crossovers, and the Daily/Weekly agreement matrix are grouped together since they all consume the same EMA columns.

**MATHEMATICAL DEFINITION** — Bullish alignment: `EMA10>EMA20>EMA89>EMA200`. Bearish: the reverse. Neither → `MIXED`. Alignment ≠ crossover: a crossover is `A[t-1]<=B[t-1] and A[t]>B[t]` (bullish) / mirrored (bearish).

In [ ]:
STATE_BULLISH_ALIGNED = "BULLISH_ALIGNED"
STATE_BEARISH_ALIGNED = "BEARISH_ALIGNED"
STATE_MIXED = "MIXED"


def classify_alignment(e10, e20, e89, e200) -> pd.Series:
    bullish = (e10 > e20) & (e20 > e89) & (e89 > e200)
    bearish = (e10 < e20) & (e20 < e89) & (e89 < e200)
    state = pd.Series(STATE_MIXED, index=e10.index)
    state[bullish] = STATE_BULLISH_ALIGNED
    state[bearish] = STATE_BEARISH_ALIGNED
    has_nan = e10.isna() | e20.isna() | e89.isna() | e200.isna()
    state[has_nan] = np.nan
    return state


def bullish_crossover(a: pd.Series, b: pd.Series) -> pd.Series:
    a_prev, b_prev = a.shift(1), b.shift(1)
    return ((a_prev <= b_prev) & (a > b)).fillna(False)


def bearish_crossover(a: pd.Series, b: pd.Series) -> pd.Series:
    a_prev, b_prev = a.shift(1), b.shift(1)
    return ((a_prev >= b_prev) & (a < b)).fillna(False)


PAIRS = [("EMA10", "EMA20"), ("EMA20", "EMA89"), ("EMA89", "EMA200"),
         ("EMA10", "EMA89"), ("EMA10", "EMA200"), ("EMA20", "EMA200")]


def compute_all_crossovers(df: pd.DataFrame, suffix: str = "") -> pd.DataFrame:
    """Section 6: does NOT assume all pairwise crossovers occur on the same day."""
    out = df.copy()
    for a_name, b_name in PAIRS:
        a, b = out[f"{a_name}{suffix}"], out[f"{b_name}{suffix}"]
        out[f"BULL_X_{a_name}_{b_name}{suffix}"] = bullish_crossover(a, b)
        out[f"BEAR_X_{a_name}_{b_name}{suffix}"] = bearish_crossover(a, b)
    return out


# Section 12: Daily-vs-Weekly agreement matrix, reported not auto-filtered
MTF_LABELS = {
    (STATE_BULLISH_ALIGNED, STATE_BULLISH_ALIGNED): "FULL_TREND_ALIGNMENT_BULL",
    (STATE_BULLISH_ALIGNED, STATE_MIXED): "WEEKLY_BULL_DAILY_PULLBACK",
    (STATE_BULLISH_ALIGNED, STATE_BEARISH_ALIGNED): "WEEKLY_BULL_DAILY_BEAR_COUNTERTREND",
    (STATE_BEARISH_ALIGNED, STATE_BEARISH_ALIGNED): "FULL_BEAR_ALIGNMENT",
    (STATE_BEARISH_ALIGNED, STATE_MIXED): "WEEKLY_BEAR_DAILY_PULLBACK",
    (STATE_BEARISH_ALIGNED, STATE_BULLISH_ALIGNED): "WEEKLY_BEAR_DAILY_BULL_COUNTERTREND",
    (STATE_MIXED, STATE_BULLISH_ALIGNED): "WEEKLY_MIXED_DAILY_BULL",
    (STATE_MIXED, STATE_BEARISH_ALIGNED): "WEEKLY_MIXED_DAILY_BEAR",
    (STATE_MIXED, STATE_MIXED): "WEEKLY_MIXED_DAILY_MIXED",
}


def compute_mtf_matrix(weekly_state: pd.Series, daily_state: pd.Series) -> pd.Series:
    pairs = list(zip(weekly_state, daily_state))
    labels = [MTF_LABELS.get(p, np.nan) for p in pairs]
    return pd.Series(labels, index=daily_state.index, name="MTF_State")


print("State + crossover + multi-timeframe-agreement engine defined.")


## Cell 10 — Cluster engine & crossover Definitions A–D (Sections 7–9)

**RESEARCH HYPOTHESIS**: compression/expansion use *rolling percentile* thresholds (not a fixed universal number), because volatility differs across stocks. Definitions A–D below are four **independent, separately testable** interpretations of \"EMA cluster crossover\" — none is asserted correct.

In [ ]:
def compute_cluster_metrics(df: pd.DataFrame, close_col: str = "Close", suffix: str = "") -> pd.DataFrame:
    out = df.copy()
    e10, e20, e89, e200 = out[f"EMA10{suffix}"], out[f"EMA20{suffix}"], out[f"EMA89{suffix}"], out[f"EMA200{suffix}"]
    stacked = pd.concat([e10, e20, e89, e200], axis=1)
    cmax, cmin = stacked.max(axis=1), stacked.min(axis=1)
    out[f"Cluster_Width_Pct{suffix}"] = (cmax - cmin) / out[close_col] * 100.0
    out[f"Cluster_Center{suffix}"] = stacked.mean(axis=1)
    for a, b in [("EMA10", "EMA20"), ("EMA20", "EMA89"), ("EMA89", "EMA200"), ("EMA10", "EMA200")]:
        av, bv = out[f"{a}{suffix}"], out[f"{b}{suffix}"]
        out[f"Dist_{a}_{b}_Pct{suffix}"] = (av - bv) / out[close_col] * 100.0
    return out


def compute_cluster_compression(df: pd.DataFrame, suffix: str = "", lookback: int = None) -> pd.DataFrame:
    if lookback is None:
        lookback = CFG.CLUSTER_WIDTH_LOOKBACK
    out = df.copy()
    width = out[f"Cluster_Width_Pct{suffix}"]
    pctl_rank = width.rolling(lookback, min_periods=30).apply(lambda w: (w[-1] <= w).mean(), raw=True)
    out[f"Cluster_Width_Percentile{suffix}"] = pctl_rank
    out[f"Cluster_Compressed_Flag{suffix}"] = pctl_rank <= CFG.CLUSTER_COMPRESSION_PCTL
    out[f"Cluster_ExpandedRegime_Flag{suffix}"] = pctl_rank >= CFG.CLUSTER_EXPANSION_PCTL
    return out


def compute_cluster_expansion(df: pd.DataFrame, suffix: str = "", lookback: int = None,
                               flat_band_pctl: float = 0.10) -> pd.DataFrame:
    if lookback is None:
        lookback = CFG.CLUSTER_WIDTH_LOOKBACK
    out = df.copy()
    width = out[f"Cluster_Width_Pct{suffix}"]
    out[f"Cluster_Width_Change_1D{suffix}"] = width.diff(1)
    out[f"Cluster_Width_Change_5D{suffix}"] = width.diff(5)

    def _slope(y):
        if np.isnan(y).any():
            return np.nan
        return np.polyfit(np.arange(len(y)), y, 1)[0]

    out[f"Cluster_Width_Slope{suffix}"] = width.rolling(5).apply(_slope, raw=True)
    change5 = out[f"Cluster_Width_Change_5D{suffix}"]
    roll = change5.rolling(lookback, min_periods=30)
    lo, hi = roll.quantile(0.5 - flat_band_pctl / 2), roll.quantile(0.5 + flat_band_pctl / 2)
    state = pd.Series("FLAT", index=out.index)
    state[change5 > hi] = "EXPANDING"
    state[change5 < lo] = "CONTRACTING"
    state[change5.isna() | lo.isna() | hi.isna()] = np.nan
    out[f"Cluster_Expansion_State{suffix}"] = state
    return out


def definition_A_sequential_crossover(df: pd.DataFrame, suffix: str = "", max_gap: int = None) -> pd.Series:
    """DEF A: EMA10xEMA20 -> EMA20xEMA89 -> EMA89xEMA200, each within max_gap sessions."""
    if max_gap is None:
        max_gap = CFG.SEQUENTIAL_CROSSOVER_MAX_GAP
    x1, x2, x3 = df[f"BULL_X_EMA10_EMA20{suffix}"], df[f"BULL_X_EMA20_EMA89{suffix}"], df[f"BULL_X_EMA89_EMA200{suffix}"]
    idx = df.index
    result = np.zeros(len(idx), dtype=bool)
    i1s, i2s, i3s = np.where(x1.to_numpy())[0], np.where(x2.to_numpy())[0], np.where(x3.to_numpy())[0]
    for i3 in i3s:
        cand2 = [i for i in i2s if 0 <= i3 - i <= max_gap]
        for i2 in cand2:
            if any(0 <= i2 - i1 <= max_gap for i1 in i1s):
                result[i3] = True
                break
    return pd.Series(result, index=idx, name=f"DefA_SequentialCrossover{suffix}")


def definition_B_alignment_transition(state: pd.Series) -> pd.Series:
    """DEF B: previous state != bullish, current state == bullish."""
    prev = state.shift(1)
    return ((prev != STATE_BULLISH_ALIGNED) & (state == STATE_BULLISH_ALIGNED)).fillna(False).rename("DefB_AlignmentTransition")


def definition_C_price_through_cluster(df: pd.DataFrame, close_col: str = "Close", suffix: str = "") -> pd.Series:
    """DEF C: price was <= cluster max previously, now > all four EMAs."""
    emas = [df[f"EMA10{suffix}"], df[f"EMA20{suffix}"], df[f"EMA89{suffix}"], df[f"EMA200{suffix}"]]
    cluster_max = pd.concat(emas, axis=1).max(axis=1)
    price = df[close_col]
    prev_below = price.shift(1) <= cluster_max.shift(1)
    now_above_all = np.logical_and.reduce([price > e for e in emas])
    return (prev_below & now_above_all).fillna(False).rename(f"DefC_PriceThroughCluster{suffix}")


def definition_D_compression_then_expansion(df: pd.DataFrame, suffix: str = "", lookahead: int = 10) -> pd.Series:
    """DEF D: cluster was compressed recently AND is now EXPANDING with a bullish directional bias."""
    compressed_recently = df[f"Cluster_Compressed_Flag{suffix}"].rolling(lookahead, min_periods=1).max().astype(bool)
    expanding_now = df[f"Cluster_Expansion_State{suffix}"] == "EXPANDING"
    directional_bull = df["Close"] > df[f"Cluster_Center{suffix}"]
    cond = compressed_recently.shift(1).fillna(False) & expanding_now & directional_bull
    return cond.fillna(False).rename(f"DefD_CompressionExpansion{suffix}")


print("Cluster engine + Definitions A-D defined.")


## Cell 11 — Slope & volatility (ATR) engine (Sections 10, 21)

*Consolidated cell*: ATR is computed here (rather than in a later 'risk' cell) because the Pullback engine (Cell 13) needs it — the 24-cell outline groups ATR under volatility, but execution order requires it before pullback/entry classification.

In [ ]:
def compute_ema_slopes(df: pd.DataFrame, suffix: str = "", k: int = 10) -> pd.DataFrame:
    out = df.copy()
    for name in ["EMA10", "EMA20", "EMA89", "EMA200"]:
        col = f"{name}{suffix}"
        out[f"{name}_Slope_Pct{suffix}"] = (out[col] / out[col].shift(k) - 1.0) * 100.0
        out[f"{name}_Slope_PerDay{suffix}"] = out[f"{name}_Slope_Pct{suffix}"] / k
    return out


def compute_atr(df: pd.DataFrame, period: int = None) -> pd.DataFrame:
    if period is None:
        period = CFG.ATR_PERIOD
    out = df.copy()
    high, low, close_prev = out["High"], out["Low"], out["Close"].shift(1)
    tr = pd.concat([(high - low), (high - close_prev).abs(), (low - close_prev).abs()], axis=1).max(axis=1)
    out[f"ATR{period}"] = tr.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    out["ATR_Pct"] = out[f"ATR{period}"] / out["Close"] * 100.0
    for name in ["EMA10", "EMA20", "EMA89", "EMA200"]:
        if name in out.columns:
            out[f"Distance_to_{name}_ATR"] = (out["Close"] - out[name]) / out[f"ATR{period}"]
    return out


print("Slope + ATR/volatility engine defined.")


## Cell 12 — Swing structure engine (Section 14)

**CRITICAL no-look-ahead rule**: a pivot at bar t is only *confirmed* `SWING_RIGHT_BARS` later. Live/backtest code must use the confirmed-at date, never the swing date itself.

In [ ]:
def detect_swing_points(df: pd.DataFrame, left: int = None, right: int = None,
                         low_col: str = "Low", high_col: str = "High") -> pd.DataFrame:
    if left is None:
        left = CFG.SWING_LEFT_BARS
    if right is None:
        right = CFG.SWING_RIGHT_BARS
    lows, highs = df[low_col].to_numpy(dtype=float), df[high_col].to_numpy(dtype=float)
    n, idx = len(df), df.index
    is_swing_low, is_swing_high = np.zeros(n, dtype=bool), np.zeros(n, dtype=bool)

    for t in range(left, n - right):
        wl = lows[t - left:t + right + 1]
        if lows[t] == wl.min() and np.sum(wl == wl.min()) == 1:
            is_swing_low[t] = True
        wh = highs[t - left:t + right + 1]
        if highs[t] == wh.max() and np.sum(wh == wh.max()) == 1:
            is_swing_high[t] = True

    out = df.copy()
    out["Is_Swing_Low"], out["Is_Swing_High"] = is_swing_low, is_swing_high
    confirmed_idx = np.minimum(np.arange(n) + right, n - 1)
    out["Swing_Low_Confirmed_At"] = idx[confirmed_idx]
    out["Swing_High_Confirmed_At"] = idx[confirmed_idx]
    not_confirmable = np.arange(n) > (n - 1 - right)
    out.loc[not_confirmable, "Is_Swing_Low"] = False
    out.loc[not_confirmable, "Is_Swing_High"] = False
    return out


def latest_confirmed_swing_low(swing_df: pd.DataFrame, as_of_date: pd.Timestamp, low_col: str = "Low") -> dict:
    mask = swing_df["Is_Swing_Low"] & (swing_df["Swing_Low_Confirmed_At"] <= as_of_date) & (swing_df.index <= as_of_date)
    eligible = swing_df[mask]
    if eligible.empty:
        return {"date": None, "price": None}
    last_date = eligible.index[-1]
    return {"date": last_date, "price": float(eligible.loc[last_date, low_col])}


def latest_confirmed_swing_high(swing_df: pd.DataFrame, as_of_date: pd.Timestamp, high_col: str = "High") -> dict:
    mask = swing_df["Is_Swing_High"] & (swing_df["Swing_High_Confirmed_At"] <= as_of_date) & (swing_df.index <= as_of_date)
    eligible = swing_df[mask]
    if eligible.empty:
        return {"date": None, "price": None}
    last_date = eligible.index[-1]
    return {"date": last_date, "price": float(eligible.loc[last_date, high_col])}


def vectorized_last_confirmed_swing_low(swing_df: pd.DataFrame, low_col: str = "Low") -> pd.Series:
    """Places each swing-low PRICE at its CONFIRMATION date (not the swing date), then ffills."""
    confirmed_value = pd.Series(np.nan, index=swing_df.index)
    for swing_date, row in swing_df[swing_df["Is_Swing_Low"]].iterrows():
        confirmed_value.loc[row["Swing_Low_Confirmed_At"]] = row[low_col]
    return confirmed_value.ffill()


def vectorized_last_confirmed_swing_high(swing_df: pd.DataFrame, high_col: str = "High") -> pd.Series:
    confirmed_value = pd.Series(np.nan, index=swing_df.index)
    for swing_date, row in swing_df[swing_df["Is_Swing_High"]].iterrows():
        confirmed_value.loc[row["Swing_High_Confirmed_At"]] = row[high_col]
    return confirmed_value.ffill()


print("Swing structure engine defined (confirmation-lagged, no look-ahead).")


## Cell 13 — Pullback / continuation engine (Section 13)

**RESEARCH HYPOTHESIS**: pullback depth = ATR-normalized distance of price *below the EMA cluster center*, conditioned on a prior breakout above the cluster. Bands are configurable in `Config` and must be validated empirically (see Cell 19), not assumed correct.

In [ ]:
PULLBACK_STATES = ["NO_PULLBACK", "SHALLOW_PULLBACK", "MODERATE_PULLBACK", "DEEP_PULLBACK", "TREND_FAILURE"]


def classify_pullback(df: pd.DataFrame, atr_col: str = "ATR14", suffix: str = "") -> pd.Series:
    center, price, atr, ema200 = df[f"Cluster_Center{suffix}"], df["Close"], df[atr_col], df[f"EMA200{suffix}"]
    was_above_cluster = price.shift(1) > center.shift(1)
    ever_broke_out = was_above_cluster.rolling(60, min_periods=1).max().astype(bool)
    dist_below_center_atr = (center - price) / atr

    state = pd.Series("NO_PULLBACK", index=df.index)
    in_pullback = ever_broke_out & (price < center)
    state[in_pullback & (dist_below_center_atr <= CFG.PULLBACK_SHALLOW_ATR)] = "SHALLOW_PULLBACK"
    state[in_pullback & (dist_below_center_atr > CFG.PULLBACK_SHALLOW_ATR) &
          (dist_below_center_atr <= CFG.PULLBACK_MODERATE_ATR)] = "MODERATE_PULLBACK"
    state[in_pullback & (dist_below_center_atr > CFG.PULLBACK_MODERATE_ATR) &
          (dist_below_center_atr <= CFG.PULLBACK_DEEP_ATR)] = "DEEP_PULLBACK"
    state[in_pullback & (price < ema200)] = "TREND_FAILURE"
    state[atr.isna() | center.isna()] = np.nan
    return state


print("Pullback engine defined.")


## Cell 14 — Entry models A–E (Section 15)

Five **independently identified, separately-tested** long-side entry definitions (reverse logic applies for short-side research, per the handwritten note's point 8). None is combined into one opaque signal.

In [ ]:
def entry_model_A_fresh_alignment(state: pd.Series) -> pd.Series:
    prev = state.shift(1)
    return ((prev != STATE_BULLISH_ALIGNED) & (state == STATE_BULLISH_ALIGNED)).fillna(False).rename("EntryA_FreshAlignment")


def entry_model_B_cluster_breakout(price: pd.Series, cluster_max: pd.Series) -> pd.Series:
    cond = (price.shift(1) <= cluster_max.shift(1)) & (price > cluster_max)
    return cond.fillna(False).rename("EntryB_ClusterBreakout")


def entry_model_C_pullback_continuation(pullback_state: pd.Series, price: pd.Series, ema20: pd.Series) -> pd.Series:
    was_pulling_back = pullback_state.shift(1).isin(["SHALLOW_PULLBACK", "MODERATE_PULLBACK"])
    reclaims = (price.shift(1) <= ema20.shift(1)) & (price > ema20)
    return (was_pulling_back & reclaims).fillna(False).rename("EntryC_PullbackContinuation")


def entry_model_D_swing_high_breakout(price: pd.Series, state: pd.Series, last_confirmed_swing_high: pd.Series) -> pd.Series:
    cond = (price.shift(1) <= last_confirmed_swing_high.shift(1)) & (price > last_confirmed_swing_high) & (state == STATE_BULLISH_ALIGNED)
    return cond.fillna(False).rename("EntryD_SwingHighBreakout")


def entry_model_E_reclaim_after_pullback(pullback_state: pd.Series, price: pd.Series, cluster_center: pd.Series, state: pd.Series) -> pd.Series:
    was_pulling_back = pullback_state.shift(1).isin(["SHALLOW_PULLBACK", "MODERATE_PULLBACK", "DEEP_PULLBACK"])
    reclaims_center = (price.shift(1) <= cluster_center.shift(1)) & (price > cluster_center)
    structure_intact = state != STATE_BEARISH_ALIGNED
    return (was_pulling_back & reclaims_center & structure_intact).fillna(False).rename("EntryE_ReclaimAfterPullback")


def compute_all_entry_models(df: pd.DataFrame, swing_df: pd.DataFrame, state: pd.Series,
                              pullback_state: pd.Series, suffix: str = "") -> pd.DataFrame:
    out = df.copy()
    price = out["Close"]
    cluster_cols = [f"EMA10{suffix}", f"EMA20{suffix}", f"EMA89{suffix}", f"EMA200{suffix}"]
    cluster_max = out[cluster_cols].max(axis=1)
    last_swing_high = vectorized_last_confirmed_swing_high(swing_df)

    out["EntryA_FreshAlignment"] = entry_model_A_fresh_alignment(state)
    out["EntryB_ClusterBreakout"] = entry_model_B_cluster_breakout(price, cluster_max)
    out["EntryC_PullbackContinuation"] = entry_model_C_pullback_continuation(pullback_state, price, out[f"EMA20{suffix}"])
    out["EntryD_SwingHighBreakout"] = entry_model_D_swing_high_breakout(price, state, last_swing_high)
    out["EntryE_ReclaimAfterPullback"] = entry_model_E_reclaim_after_pullback(pullback_state, price, out[f"Cluster_Center{suffix}"], state)
    return out


print("Entry models A-E defined.")


## Cell 15 — Market regime (Section 19)

**OPTIONAL FILTER / diagnostic first**: regime is reported for segmentation, not auto-applied as a filter, so Cell 19 can answer *whether* the setup behaves differently by regime (Hypothesis H8).

In [ ]:
def compute_market_regime(index_df: pd.DataFrame, close_col: str = "Close") -> pd.DataFrame:
    out = index_df.copy()
    out["Index_EMA20"] = compute_ema(out[close_col], CFG.INDEX_EMA_FAST)
    out["Index_EMA50"] = compute_ema(out[close_col], CFG.INDEX_EMA_MEDIUM)
    out["Index_EMA89"] = compute_ema(out[close_col], CFG.INDEX_EMA_STRUCTURAL)
    out["Index_EMA200"] = compute_ema(out[close_col], CFG.INDEX_EMA_LONG)
    price = out[close_col]
    bull = (price > out["Index_EMA50"]) & (out["Index_EMA50"] > out["Index_EMA200"])
    bear = (price < out["Index_EMA50"]) & (out["Index_EMA50"] < out["Index_EMA200"])
    regime = pd.Series("NEUTRAL", index=out.index)
    regime[bull], regime[bear] = "BULL", "BEAR"
    regime[out["Index_EMA200"].isna()] = np.nan
    out["Market_Regime"] = regime
    return out[["Market_Regime", "Index_EMA20", "Index_EMA50", "Index_EMA89", "Index_EMA200"]]


def attach_market_regime_asof(stock_df: pd.DataFrame, regime_df: pd.DataFrame) -> pd.Series:
    return regime_df["Market_Regime"].reindex(stock_df.index, method="ffill")


print("Market regime engine defined.")


## Cell 16 — Relative strength (Section 20)

In [ ]:
def compute_relative_strength(stock_close: pd.Series, index_close_aligned: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=stock_close.index)
    for window, label in [(20, "RS_20D"), (60, "RS_60D"), (120, "RS_120D")]:
        stock_ret = stock_close / stock_close.shift(window) - 1.0
        index_ret = index_close_aligned / index_close_aligned.shift(window) - 1.0
        out[label] = (stock_ret - index_ret) * 100.0
    return out


def cross_sectional_percentile(rs_series_today: pd.Series) -> pd.Series:
    """Percentile rank of each stock's RS within the universe on a given day."""
    return rs_series_today.rank(pct=True) * 100.0


print("Relative strength engine defined.")


## Cell 17 — Risk engine: stops, position sizing, execution, costs (Sections 16, 17, 18, 31)

Position sizing is kept **fully separate** from signal generation (Section 17). Execution is modeled explicitly and separately from signal detection (Section 18) — no silent same-close fill assumption.

In [ ]:
def compute_stops(df: pd.DataFrame, swing_df: pd.DataFrame, atr_col: str = "ATR14") -> pd.DataFrame:
    out = df.copy()
    out["Swing_Low_Stop"] = vectorized_last_confirmed_swing_low(swing_df)
    out["Swing_High_Stop"] = vectorized_last_confirmed_swing_high(swing_df)
    out["Stop_Distance_Pct_Long"] = (out["Close"] - out["Swing_Low_Stop"]) / out["Close"] * 100.0
    out["Stop_Distance_ATR_Long"] = (out["Close"] - out["Swing_Low_Stop"]) / out[atr_col]
    out["ATR_Based_Stop_Long"] = out["Close"] - CFG.ATR_STOP_MULTIPLE * out[atr_col]   # OPTIONAL, kept separate
    out["Stop_Distance_Pct_Short"] = (out["Swing_High_Stop"] - out["Close"]) / out["Close"] * 100.0
    out["Stop_Distance_ATR_Short"] = (out["Swing_High_Stop"] - out["Close"]) / out[atr_col]
    out["ATR_Based_Stop_Short"] = out["Close"] + CFG.ATR_STOP_MULTIPLE * out[atr_col]
    return out


def position_size(capital: float, entry_price: float, stop_price: float,
                   risk_per_trade_pct: float = None, lot_size: int = 1,
                   min_qty: int = 1, max_qty: int = None) -> dict:
    if risk_per_trade_pct is None:
        risk_per_trade_pct = CFG.RISK_PER_TRADE_PCT
    risk_capital = capital * (risk_per_trade_pct / 100.0)
    per_share_risk = abs(entry_price - stop_price)
    if per_share_risk <= 0 or np.isnan(per_share_risk):
        return {"qty": 0, "risk_capital": risk_capital, "reason": "invalid_stop_distance"}
    lots = max(int((risk_capital / per_share_risk) // lot_size), 0)
    qty = max(lots * lot_size, 0)
    if qty < min_qty:
        return {"qty": 0, "risk_capital": risk_capital, "reason": "below_min_qty"}
    if max_qty is not None:
        qty = min(qty, max_qty)
    return {"qty": qty, "risk_capital": risk_capital, "per_share_risk": per_share_risk, "reason": "ok"}


def resolve_execution_price(df: pd.DataFrame, signal_date: pd.Timestamp, execution_model: str = None) -> dict:
    """SIGNAL DETECTION uses signal_date's close; EXECUTION is a separate,
    explicit assumption: 'same_close' | 'next_open' (default) | 'next_close'."""
    if execution_model is None:
        execution_model = CFG.EXECUTION_MODEL
    idx = df.index
    loc = idx.get_loc(signal_date)
    if execution_model == "same_close":
        return {"date": signal_date, "price": float(df.loc[signal_date, "Close"]), "model": execution_model}
    if loc + 1 >= len(idx):
        return {"date": None, "price": None, "model": execution_model, "reason": "no_next_bar_available"}
    next_date = idx[loc + 1]
    if execution_model == "next_open":
        return {"date": next_date, "price": float(df.loc[next_date, "Open"]), "model": execution_model}
    if execution_model == "next_close":
        return {"date": next_date, "price": float(df.loc[next_date, "Close"]), "model": execution_model}
    raise ValueError(f"Unknown execution_model: {execution_model}")


def compute_transaction_cost_pct() -> float:
    """Section 31: one-leg cost fraction. PLACEHOLDER values in Config - user must confirm."""
    taxable = CFG.BROKERAGE_PCT + CFG.EXCHANGE_CHARGES_PCT
    gst = taxable * CFG.GST_PCT
    total = CFG.BROKERAGE_PCT + CFG.STT_PCT + CFG.EXCHANGE_CHARGES_PCT + gst + CFG.SLIPPAGE_PCT
    return total / 100.0


print("Risk engine defined (stops, position sizing, execution model, transaction costs).")
print("One-way transaction cost (fraction):", compute_transaction_cost_pct())


## Cell 18 — Signal state machine, scoring, master pipeline & Live Scanner (Mode 1)

This is the largest cell: it wires every prior engine into ONE audited per-stock feature pipeline (`build_stock_feature_frame`), then runs **Mode 1 — Live Scan** across the universe. The same pipeline is reused unchanged by Mode 2 (Cell 19) and Mode 3 (Cell 20), so all three modes evaluate identical feature definitions.

**EMPIRICALLY TESTED RESULT** disclaimer: `Signal_Reason` (Section 33) never asserts \"high probability\" — only what Sections 25–26 actually measure would justify that language, and only after inspection.

In [ ]:
SIGNAL_STATES = ["WATCH", "TREND_FORMING", "BULLISH_CLUSTER", "PULLBACK", "ENTRY_TRIGGERED",
                  "IN_POSITION_RESEARCH_STATE", "TREND_CONTINUATION", "TREND_FAILURE",
                  "EXIT_TRIGGER", "INVALIDATED"]


def compute_signal_state(daily_state, pullback_state, any_entry_triggered, cluster_expansion) -> pd.Series:
    """Deterministic, fully-reproducible lifecycle label (Section 23) - not a hidden ML classifier."""
    out = pd.Series("WATCH", index=daily_state.index)
    out[daily_state == STATE_BULLISH_ALIGNED] = "TREND_FORMING"
    out[(daily_state == STATE_BULLISH_ALIGNED) & (cluster_expansion == "EXPANDING")] = "BULLISH_CLUSTER"
    out[pullback_state.isin(["SHALLOW_PULLBACK", "MODERATE_PULLBACK", "DEEP_PULLBACK"])] = "PULLBACK"
    out[pullback_state == "TREND_FAILURE"] = "TREND_FAILURE"
    out[any_entry_triggered] = "ENTRY_TRIGGERED"
    out[daily_state.isna()] = np.nan
    return out


def compute_score_components(row: pd.Series) -> dict:
    """Section 24: components kept SEPARATE; composite below is explicitly a heuristic."""
    scores = {}
    scores["Trend_Alignment_Score"] = 1.0 if row.get("Daily_State") == STATE_BULLISH_ALIGNED else (0.5 if row.get("Daily_State") == STATE_MIXED else 0.0)
    scores["MultiTimeframe_Score"] = 1.0 if str(row.get("MTF_State", "")).startswith("FULL_TREND_ALIGNMENT") else (0.5 if "PULLBACK" in str(row.get("MTF_State", "")) else 0.0)
    exp_state = row.get("Cluster_Expansion_State")
    scores["Cluster_Expansion_Score"] = 1.0 if exp_state == "EXPANDING" else (0.5 if exp_state == "FLAT" else 0.0)
    slope = row.get("EMA20_Slope_Pct", np.nan)
    scores["Momentum_Score"] = float(np.clip((slope + 5) / 10, 0, 1)) if pd.notna(slope) else np.nan
    rs = row.get("RS_60D", np.nan)
    scores["RelativeStrength_Score"] = float(np.clip((rs + 10) / 20, 0, 1)) if pd.notna(rs) else np.nan
    turnover_ratio = row.get("Volume_Ratio", np.nan)
    scores["Liquidity_Score"] = float(np.clip(turnover_ratio, 0, 2) / 2) if pd.notna(turnover_ratio) else np.nan
    pb = row.get("Pullback_State")
    scores["Pullback_Quality_Score"] = 1.0 if pb == "SHALLOW_PULLBACK" else (0.6 if pb == "MODERATE_PULLBACK" else (0.2 if pb == "DEEP_PULLBACK" else 0.0))
    return scores


def compute_composite_heuristic_score(scores: dict, weights: dict = None) -> float:
    """EXPLICITLY a heuristic (Section 24): not a statistically validated ranking."""
    if weights is None:
        weights = {k: 1.0 / len(scores) for k in scores}
    vals = [(scores[k] * weights.get(k, 0)) for k in scores if pd.notna(scores[k])]
    wsum = sum(weights.get(k, 0) for k in scores if pd.notna(scores[k]))
    return float(sum(vals) / wsum) if wsum else np.nan


def generate_signal_reason(row: pd.Series) -> str:
    """Section 33: human-readable, no unsupported 'high probability' language."""
    parts = []
    if row.get("Daily_State") == STATE_BULLISH_ALIGNED:
        parts.append("Daily EMA structure is bullish (EMA10>EMA20>EMA89>EMA200).")
    elif row.get("Daily_State") == STATE_BEARISH_ALIGNED:
        parts.append("Daily EMA structure is bearish (EMA10<EMA20<EMA89<EMA200).")
    else:
        parts.append("Daily EMA structure is mixed / not fully aligned.")
    if row.get("Weekly_State") == STATE_BULLISH_ALIGNED:
        parts.append("Weekly EMA structure is also bullish.")
    elif row.get("Weekly_State") == STATE_BEARISH_ALIGNED:
        parts.append("Weekly EMA structure is bearish.")
    exp_state = row.get("Cluster_Expansion_State")
    if exp_state == "EXPANDING":
        parts.append("EMA cluster width is expanding.")
    elif exp_state == "CONTRACTING":
        parts.append("EMA cluster is compressing.")
    if row.get("Price_vs_EMA200") is True:
        parts.append("Price is above EMA200.")
    elif row.get("Price_vs_EMA200") is False:
        parts.append("Price is below EMA200.")
    rs = row.get("RS_60D")
    if pd.notna(rs):
        parts.append(f"60-day relative strength vs index is {'positive' if rs > 0 else 'negative'} ({rs:.1f}%).")
    pb = row.get("Pullback_State")
    if pb in ("SHALLOW_PULLBACK", "MODERATE_PULLBACK"):
        parts.append(f"Stock is in a {pb.replace('_', ' ').lower()} toward the EMA cluster.")
    if row.get("EntryD_SwingHighBreakout"):
        parts.append("A confirmed break of the most recent swing high has occurred.")
    return " ".join(parts) if parts else "Insufficient data to generate a reason."


def build_diagnostics_row(ticker: str, row: pd.Series) -> dict:
    """Section 34: full auditability - pass/fail/state for every stage."""
    return {
        "Ticker": ticker, "Date": row.name,
        "Daily_EMA_Alignment": row.get("Daily_State"), "Weekly_EMA_Alignment": row.get("Weekly_State"),
        "EMA200_Condition_PriceAbove": row.get("Price_vs_EMA200"),
        "Cluster_State": row.get("Cluster_Expansion_State"), "Cluster_Width_Pct": row.get("Cluster_Width_Pct"),
        "Price_vs_Cluster": "ABOVE" if row.get("Close", np.nan) > row.get("Cluster_Center", np.nan) else "BELOW",
        "Pullback_Status": row.get("Pullback_State"), "Swing_Low_Stop": row.get("Swing_Low_Stop"),
        "Relative_Strength_60D": row.get("RS_60D"), "Market_Regime": row.get("Market_Regime"),
        "Liquidity_Volume_Ratio": row.get("Volume_Ratio"), "Final_Entry_Condition": row.get("Signal_State"),
    }


def build_state_change_log(signal_state: pd.Series, ticker: str) -> pd.DataFrame:
    """Section 39."""
    prev = signal_state.shift(1)
    changed = (signal_state != prev) & signal_state.notna() & prev.notna()
    rows = [{"Ticker": ticker, "Date": d, "Previous_State": prev.loc[d], "New_State": signal_state.loc[d],
             "Event": f"{prev.loc[d]} -> {signal_state.loc[d]}"} for d in signal_state.index[changed]]
    return pd.DataFrame(rows, columns=["Ticker", "Date", "Previous_State", "New_State", "Event"])


def build_stock_feature_frame(daily_ohlc: pd.DataFrame, index_close_aligned: pd.Series,
                               regime_df: pd.DataFrame, cfg: Config = None) -> pd.DataFrame:
    """THE master per-stock feature pipeline, reused unchanged by Modes 1/2/3."""
    if cfg is None:
        cfg = CFG
    daily = daily_ohlc.sort_index().copy()

    d = compute_all_emas(daily, suffix="", cfg=cfg)
    for name in ["EMA10", "EMA20", "EMA89", "EMA200"]:
        d[f"{name}_D"] = d[name]

    weekly = build_true_weekly_ohlc(daily)
    w = compute_all_emas(weekly, suffix="_W", cfg=cfg)
    w = w.rename(columns={"Open": "Open_W", "High": "High_W", "Low": "Low_W", "Close": "Close_W"})
    weekly_cols_present = [c for c in ["EMA10_W", "EMA20_W", "EMA89_W", "EMA200_W", "Open_W", "High_W", "Low_W", "Close_W"] if c in w.columns]
    d = attach_last_known_weekly(d, w, weekly_cols_present)

    d = compute_all_crossovers(d, suffix="")
    d["Daily_State"] = classify_alignment(d["EMA10"], d["EMA20"], d["EMA89"], d["EMA200"])
    d["Weekly_State"] = classify_alignment(d["EMA10_W"], d["EMA20_W"], d["EMA89_W"], d["EMA200_W"])
    d["MTF_State"] = compute_mtf_matrix(d["Weekly_State"], d["Daily_State"])

    d = compute_cluster_metrics(d, suffix="")
    d = compute_cluster_compression(d, suffix="")
    d = compute_cluster_expansion(d, suffix="")
    d = compute_ema_slopes(d, suffix="")

    d["DefA_SequentialCrossover"] = definition_A_sequential_crossover(d, suffix="")
    d["DefB_AlignmentTransition"] = definition_B_alignment_transition(d["Daily_State"])
    d["DefC_PriceThroughCluster"] = definition_C_price_through_cluster(d, suffix="")
    d["DefD_CompressionExpansion"] = definition_D_compression_then_expansion(d, suffix="")

    d = compute_atr(d, period=cfg.ATR_PERIOD)
    swing = detect_swing_points(d, left=cfg.SWING_LEFT_BARS, right=cfg.SWING_RIGHT_BARS)
    d["Is_Swing_Low"], d["Is_Swing_High"] = swing["Is_Swing_Low"], swing["Is_Swing_High"]
    d["Pullback_State"] = classify_pullback(d, atr_col=f"ATR{cfg.ATR_PERIOD}", suffix="")

    d = compute_all_entry_models(d, swing, d["Daily_State"], d["Pullback_State"], suffix="")
    entry_cols = ["EntryA_FreshAlignment", "EntryB_ClusterBreakout", "EntryC_PullbackContinuation",
                  "EntryD_SwingHighBreakout", "EntryE_ReclaimAfterPullback"]
    d["Any_Entry_Triggered"] = d[entry_cols].any(axis=1)

    d = compute_stops(d, swing, atr_col=f"ATR{cfg.ATR_PERIOD}")

    d["Price_vs_EMA200"] = d["Close"] > d["EMA200"]
    d["EMA89_vs_EMA200"] = d["EMA89"] > d["EMA200"]           # OPTIONAL FILTER, reported not enforced
    d["EMA200_Rising"] = d["EMA200"].diff(5) > 0              # OPTIONAL FILTER, reported not enforced

    d["Market_Regime"] = attach_market_regime_asof(d, regime_df)
    rs = compute_relative_strength(d["Close"], index_close_aligned.reindex(d.index, method="ffill"))
    d = pd.concat([d, rs], axis=1)

    d["Volume20"] = d["Volume"].rolling(20).mean()
    d["Volume50"] = d["Volume"].rolling(50).mean()
    d["Volume_Ratio"] = d["Volume"] / d["Volume20"]
    d["Dollar_Turnover"] = d["Close"] * d["Volume"]
    d["Average_Daily_Turnover"] = d["Dollar_Turnover"].rolling(20).mean()
    if cfg.ENABLE_LIQUIDITY_FILTER:                            # Section 22: OFF by default
        d = d[d["Average_Daily_Turnover"] >= cfg.MIN_AVG_DOLLAR_TURNOVER]

    d["Signal_State"] = compute_signal_state(d["Daily_State"], d["Pullback_State"], d["Any_Entry_Triggered"], d["Cluster_Expansion_State"])
    return d


ENTRY_MODEL_COLS = {
    "EntryA_FreshAlignment": "Immediate crossover / fresh alignment",
    "EntryB_ClusterBreakout": "Price-through-cluster breakout",
    "EntryC_PullbackContinuation": "Pullback continuation",
    "EntryD_SwingHighBreakout": "Swing-high breakout",
    "EntryE_ReclaimAfterPullback": "Reclaim after pullback",
}


def build_scan_row(ticker: str, d: pd.DataFrame, as_of_date: pd.Timestamp) -> dict:
    """Section 32: the live-scanner output row."""
    if as_of_date not in d.index:
        return None
    row = d.loc[as_of_date].copy()
    scores = compute_score_components(row)
    composite = compute_composite_heuristic_score(scores)
    reason = generate_signal_reason(row)
    triggered = [m for m in ENTRY_MODEL_COLS if bool(row.get(m))]
    out = {
        "Stock": ticker, "Signal_Date": as_of_date,
        "Daily_EMA10": row.get("EMA10"), "Daily_EMA20": row.get("EMA20"), "Daily_EMA89": row.get("EMA89"), "Daily_EMA200": row.get("EMA200"),
        "Weekly_EMA10": row.get("EMA10_W"), "Weekly_EMA20": row.get("EMA20_W"), "Weekly_EMA89": row.get("EMA89_W"), "Weekly_EMA200": row.get("EMA200_W"),
        "Daily_State": row.get("Daily_State"), "Weekly_State": row.get("Weekly_State"),
        "Cluster_Width_Pct": row.get("Cluster_Width_Pct"), "Cluster_Width_Change": row.get("Cluster_Width_Change_1D"),
        "EMA_Slope": row.get("EMA20_Slope_Pct"), "Price_vs_EMA200": row.get("Price_vs_EMA200"),
        "RS_60D": row.get("RS_60D"), "ATR_Pct": row.get("ATR_Pct"),
        "Swing_Low": row.get("Swing_Low_Stop"), "Suggested_Stop_Level": row.get("Swing_Low_Stop"),
        "Stop_Distance_Pct": row.get("Stop_Distance_Pct_Long"),
        "Entry_Model": ",".join(triggered) if triggered else "NONE",
        "Signal_State": row.get("Signal_State"), "Research_Score_Heuristic": composite, "Signal_Reason": reason,
    }
    out.update({f"Score_{k}": v for k, v in scores.items()})
    return out


def run_full_universe_pipeline(universe: dict, index_df: pd.DataFrame, cfg: Config = None):
    """Error-isolated, data-quality-gated feature build across every ticker (Sections 35/36)."""
    if cfg is None:
        cfg = CFG
    error_log = ScanErrorLog()
    dq_rows = []
    feature_frames = {}
    regime_df = compute_market_regime(index_df)
    for ticker, daily_df in universe.items():
        dq = validate_ohlc(daily_df, ticker, max_stale_days=10_000, min_history_days=260)
        dq_rows.append(dq)
        if not dq["passed"]:
            error_log.log(ticker, "data_validation", ValueError(f"Failed DQ checks: {dq['issues']}"))
            continue
        feats = run_stage_safely(ticker, "feature_pipeline", build_stock_feature_frame, error_log,
                                  daily_df, index_df["Close"], regime_df, cfg)
        if feats is not None:
            feature_frames[ticker] = feats
    dq_df = pd.DataFrame([{"ticker": r["ticker"], "passed": r["passed"], "issues": "; ".join(r["issues"])} for r in dq_rows])
    return feature_frames, error_log.to_df(), dq_df


def run_live_scan(feature_frames: dict, as_of_date: pd.Timestamp = None) -> pd.DataFrame:
    rows = []
    for ticker, feats in feature_frames.items():
        d = as_of_date if as_of_date is not None else feats.index.max()
        if d in feats.index:
            row = build_scan_row(ticker, feats, d)
            if row is not None:
                rows.append(row)
    return pd.DataFrame(rows)


def generate_synthetic_demo_universe(n_stocks: int = 8, n_days: int = 2000, seed: int = 42):
    """LOUDLY LABELED FAKE DATA - see Cell 1's data-access note. Never used for real conclusions."""
    warnings.warn(f"*** {SYNTHETIC_DEMO_TAG} *** Live data unavailable; generating a synthetic "
                  f"demo universe purely to exercise the notebook end-to-end.", stacklevel=2)
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=n_days)
    n = len(dates)  # bdate_range can return n_days-1 if `end` itself falls on a weekend; always trust len(dates)
    idx_close = 20000 + np.cumsum(rng.normal(2.0, 40, n))
    index_df = pd.DataFrame({"Close": idx_close}, index=dates)
    universe = {}
    for i in range(n_stocks):
        tk = f"{SYNTHETIC_DEMO_TAG}_{i:02d}.DEMO"
        drift = rng.normal(0.03 + 0.01 * (i % 3), 1.2 + 0.1 * i, n)
        close = 100 + np.cumsum(drift)
        openp = close + rng.normal(0, 0.3, n)
        high = np.maximum(close, openp) + rng.uniform(0.1, 1.0, n)
        low = np.minimum(close, openp) - rng.uniform(0.1, 1.0, n)
        universe[tk] = pd.DataFrame({"Open": openp, "High": high, "Low": low, "Close": close,
                                      "Volume": rng.integers(50_000, 800_000, n)}, index=dates)
    return universe, index_df


# --------------------------- MODE 1 EXECUTION ---------------------------
IS_SYNTHETIC_DEMO = False
try:
    nifty200_df = fetch_nifty200_universe()
    symbols = nifty200_df["YF_Symbol"].tolist()
    print(f"Fetched {len(symbols)} NIFTY 200 symbols from NSE.")
    start_date = (pd.Timestamp.today() - pd.DateOffset(years=CFG.MIN_HISTORY_YEARS + 1)).date().isoformat()
    ohlc_data = batch_download(symbols, start=start_date)
    universe = {s: df for s, df in ohlc_data.items() if df is not None}
    index_data = get_daily_ohlc("^NSEI", start=start_date)  # NIFTY 50 as benchmark index
    index_df = index_data[["Close"]]
    universe_meta_df = nifty200_df
    if len(universe) < 20:
        raise RuntimeError(f"Only {len(universe)} tickers downloaded successfully - aborting live universe use.")
except Exception as e:
    print(f"Live NIFTY200/yfinance path unavailable ({type(e).__name__}: {e}).")
    print("Falling back to the SYNTHETIC DEMO universe (see Cell 1 data-access note).")
    IS_SYNTHETIC_DEMO = True
    universe, index_df = generate_synthetic_demo_universe()
    universe_meta_df = pd.DataFrame({"Symbol": list(universe.keys()), "YF_Symbol": list(universe.keys())})

UNIVERSE_MODE_LABEL = SYNTHETIC_DEMO_TAG if IS_SYNTHETIC_DEMO else label_universe_mode(historical_membership_available=False)
print("Universe mode:", UNIVERSE_MODE_LABEL)

feature_frames, error_log_df, dq_report_df = run_full_universe_pipeline(universe, index_df)
print(f"\nFeature pipeline succeeded for {len(feature_frames)}/{len(universe)} tickers.")
if not error_log_df.empty:
    print(f"{len(error_log_df)} tickers failed (see error_log_df) - the rest of the scan proceeded normally.")

scan_df = run_live_scan(feature_frames)
diagnostics_df = pd.DataFrame([build_diagnostics_row(t, feature_frames[t].iloc[-1]) for t in feature_frames])
state_log_df = pd.concat([build_state_change_log(feature_frames[t]["Signal_State"], t) for t in feature_frames],
                          ignore_index=True) if feature_frames else pd.DataFrame()

print(f"\n=== MODE 1: LIVE SCAN ({'SYNTHETIC DEMO' if IS_SYNTHETIC_DEMO else 'LIVE'}) ===")
print(f"Stocks scanned: {len(feature_frames)} | Data failures: {len(error_log_df)}")
if not scan_df.empty:
    print(f"Bullish daily state: {(scan_df['Daily_State'] == STATE_BULLISH_ALIGNED).sum()}")
    print(f"Any entry model triggered today: {(scan_df['Entry_Model'] != 'NONE').sum()}")
    display_cols = ["Stock", "Daily_State", "Weekly_State", "Cluster_Width_Pct", "Entry_Model", "Signal_State"]
    print(scan_df[display_cols].to_string(index=False))


## Cell 19 — Historical event study (Mode 2) (Sections 25–26, 46–48, 50–51)

**EMPIRICALLY TESTED RESULT**: forward returns, MFE/MAE, and performance statistics for every entry model, broken down by market regime (mandatory, Section 48) and checked for single-stock concentration (Section 51). Hypotheses H1–H10 (Section 46) are evaluated from these same numbers — none are assumed true beforehand.

In [ ]:
FORWARD_HORIZONS = [1, 3, 5, 10, 20, 40]


def event_study_forward_returns(price_df: pd.DataFrame, event_dates, entry_prices: pd.Series) -> pd.DataFrame:
    idx = price_df.index
    rows = []
    max_h = max(FORWARD_HORIZONS)
    for event_date in event_dates:
        if event_date not in entry_prices.index or pd.isna(entry_prices.loc[event_date]) or event_date not in idx:
            continue
        entry_price = entry_prices.loc[event_date]
        loc = idx.get_loc(event_date)
        row = {"Event_Date": event_date, "Entry_Price": entry_price}
        path = price_df.iloc[loc:min(loc + max_h, len(idx) - 1) + 1]
        if len(path) > 1:
            row["MFE_Pct"] = (path["High"].iloc[1:].max() - entry_price) / entry_price * 100.0
            row["MAE_Pct"] = (path["Low"].iloc[1:].min() - entry_price) / entry_price * 100.0
            row["Time_to_MFE"] = int(path["High"].iloc[1:].to_numpy().argmax()) + 1
            row["Time_to_MAE"] = int(path["Low"].iloc[1:].to_numpy().argmin()) + 1
        else:
            row["MFE_Pct"] = row["MAE_Pct"] = row["Time_to_MFE"] = row["Time_to_MAE"] = np.nan
        for h in FORWARD_HORIZONS:
            fut_loc = loc + h
            row[f"Fwd_Ret_{h}D"] = (price_df["Close"].iloc[fut_loc] - entry_price) / entry_price * 100.0 if fut_loc < len(idx) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def apply_transaction_costs(event_study_df: pd.DataFrame) -> pd.DataFrame:
    out = event_study_df.copy()
    round_trip_cost_pct = compute_transaction_cost_pct() * 100.0 * 2.0
    for h in FORWARD_HORIZONS:
        col = f"Fwd_Ret_{h}D"
        if col in out.columns:
            out[f"{col}_Net"] = out[col] - round_trip_cost_pct
    out.attrs["round_trip_cost_pct"] = round_trip_cost_pct
    return out


def performance_stats(returns_pct: pd.Series) -> dict:
    r = returns_pct.dropna()
    if len(r) == 0:
        return {"n_trades": 0}
    wins, losses = r[r > 0], r[r <= 0]
    gross_win, gross_loss = wins.sum(), -losses.sum()
    profit_factor = (gross_win / gross_loss) if gross_loss > 0 else np.nan
    equity_curve = (1 + r / 100.0).cumprod()
    drawdown = (equity_curve / equity_curve.cummax() - 1.0) * 100.0
    return {
        "n_trades": int(len(r)), "win_rate_pct": float(len(wins) / len(r) * 100.0),
        "avg_return_pct": float(r.mean()), "median_return_pct": float(r.median()),
        "std_return_pct": float(r.std()), "profit_factor": float(profit_factor) if pd.notna(profit_factor) else np.nan,
        "expectancy_pct": float(r.mean()), "avg_win_pct": float(wins.mean()) if len(wins) else np.nan,
        "avg_loss_pct": float(losses.mean()) if len(losses) else np.nan,
        "best_trade_pct": float(r.max()), "worst_trade_pct": float(r.min()),
        "max_drawdown_pct": float(drawdown.min()), "skew": float(r.skew()) if len(r) > 2 else np.nan,
    }


def performance_stats_by_group(event_df: pd.DataFrame, return_col: str, group_col: str) -> pd.DataFrame:
    if group_col not in event_df.columns:
        return pd.DataFrame()
    rows = []
    for g, sub in event_df.groupby(group_col, dropna=True):
        stats = performance_stats(sub[return_col]); stats[group_col] = g
        rows.append(stats)
    return pd.DataFrame(rows)


def run_event_study_all_models(feature_frames: dict, execution_model: str = None) -> pd.DataFrame:
    all_rows = []
    for model_col, model_label in ENTRY_MODEL_COLS.items():
        for ticker, feats in feature_frames.items():
            event_dates = feats.index[feats[model_col].fillna(False)]
            if len(event_dates) == 0:
                continue
            entry_prices = {}
            for ed in event_dates:
                exe = resolve_execution_price(feats, ed, execution_model=execution_model)
                if exe.get("price") is not None:
                    entry_prices[ed] = exe["price"]
            if not entry_prices:
                continue
            es = event_study_forward_returns(feats, pd.Series(entry_prices).index, pd.Series(entry_prices))
            if es.empty:
                continue
            es["Ticker"], es["Entry_Model"], es["Entry_Model_Label"] = ticker, model_col, model_label
            es["Market_Regime"] = feats.loc[es["Event_Date"], "Market_Regime"].to_numpy()
            all_rows.append(es)
    return apply_transaction_costs(pd.concat(all_rows, ignore_index=True)) if all_rows else pd.DataFrame()


def comparison_table(event_df: pd.DataFrame, horizon: int = 20) -> pd.DataFrame:
    """Section 47."""
    if event_df.empty:
        return pd.DataFrame()
    col, col_net = f"Fwd_Ret_{horizon}D", f"Fwd_Ret_{horizon}D_Net"
    rows = []
    for model_col, model_label in ENTRY_MODEL_COLS.items():
        sub = event_df[event_df["Entry_Model"] == model_col]
        if sub.empty:
            continue
        stats, stats_net = performance_stats(sub[col]), performance_stats(sub[col_net]) if col_net in sub.columns else {}
        rows.append({
            "Entry_Definition": model_label, "N_Occurrences": stats.get("n_trades", 0),
            f"{horizon}D_Win_Rate_Pct": stats.get("win_rate_pct"), f"{horizon}D_Median_Return_Pct": stats.get("median_return_pct"),
            f"{horizon}D_Avg_Return_Pct_Gross": stats.get("avg_return_pct"), f"{horizon}D_Avg_Return_Pct_Net": stats_net.get("avg_return_pct"),
            "Max_Favorable_Excursion_Pct": sub["MFE_Pct"].mean(), "Max_Adverse_Excursion_Pct": sub["MAE_Pct"].mean(),
            "Max_Drawdown_Pct": stats.get("max_drawdown_pct"),
        })
    return pd.DataFrame(rows)


def regime_breakdown(event_df: pd.DataFrame, horizon: int = 20) -> pd.DataFrame:
    """Section 48: MANDATORY bull/neutral/bear breakdown per entry model."""
    if event_df.empty:
        return pd.DataFrame()
    col = f"Fwd_Ret_{horizon}D"
    rows = []
    for model_col, model_label in ENTRY_MODEL_COLS.items():
        for regime in ["BULL", "NEUTRAL", "BEAR"]:
            sub = event_df[(event_df["Entry_Model"] == model_col) & (event_df["Market_Regime"] == regime)]
            if sub.empty:
                continue
            stats = performance_stats(sub[col]); stats["Entry_Definition"], stats["Regime"] = model_label, regime
            rows.append(stats)
    return pd.DataFrame(rows)


def stock_concentration_check(event_df: pd.DataFrame) -> pd.DataFrame:
    """Section 51: is the result driven by one stock?"""
    if event_df.empty:
        return pd.DataFrame()
    return (event_df.groupby(["Entry_Model", "Ticker"]).size().rename("n_signals").reset_index()
            .sort_values(["Entry_Model", "n_signals"], ascending=[True, False]))


# --------------------------- MODE 2 EXECUTION ---------------------------
event_df = run_event_study_all_models(feature_frames)
comparison_df = comparison_table(event_df, horizon=20)
regime_breakdown_df = regime_breakdown(event_df, horizon=20)
concentration_df = stock_concentration_check(event_df)

print(f"=== MODE 2: HISTORICAL EVENT STUDY ({'SYNTHETIC DEMO' if IS_SYNTHETIC_DEMO else 'LIVE'}) ===")
print(f"Total events across all 5 entry models: {len(event_df)}\n")
print("Comparison table (Section 47):")
print(comparison_df.to_string(index=False) if not comparison_df.empty else "  (no events found)")
print("\nRegime breakdown, first 10 rows (Section 48 - mandatory):")
print(regime_breakdown_df.head(10).to_string(index=False) if not regime_breakdown_df.empty else "  (no events found)")
print("\nStock concentration, first 10 rows (Section 51):")
print(concentration_df.head(10).to_string(index=False) if not concentration_df.empty else "  (no events found)")

print("""
Hypotheses H1-H10 (Section 46) - inspect the tables above to evaluate each;
none is assumed true:
  H1  Bullish alignment predicts positive forward returns          -> see EntryA row, comparison_df
  H2  Daily+Weekly bullish beats daily-only                        -> filter event_df by MTF_State and compare
  H3  Cluster expansion-after-compression is more informative      -> compare DefD-tagged events vs raw alignment
  H4  Breakout-from-below differs from mature-trend-already-above  -> compare EntryB vs EntryA cohorts
  H5  Pullback-continuation differs from immediate crossover       -> compare EntryC/E vs EntryA in comparison_df
  H6  Positive relative strength improves the setup                -> segment event_df by RS_60D sign
  H7  Rising EMA89/EMA200 improves continuation quality             -> segment by EMA200_Rising
  H8  Market regime changes expectancy                             -> regime_breakdown_df (mandatory check)
  H9  ATR-normalized distance improves risk management             -> compare Stop_Distance_ATR distributions
  H10 Swing-breakout entries improve timing vs immediate crossover -> compare EntryD vs EntryA in comparison_df
""")


## Cell 20 — Walk-forward research (Mode 3) (Section 28)

Splits history into rolling train/test windows; every reported statistic uses **only** price action inside its own test window - never the same data used to pick anything. This is the honest out-of-sample check the in-sample event study (Cell 19) cannot provide by itself.

In [ ]:
def walk_forward_splits(dates: pd.DatetimeIndex, train_years: float = 3.0,
                         test_years: float = 1.0, step_years: float = 1.0):
    start, end = dates.min(), dates.max()
    splits, train_start = [], start
    while True:
        train_end = train_start + pd.DateOffset(years=train_years)
        test_start = train_end
        test_end = test_start + pd.DateOffset(years=test_years)
        if test_start >= end:
            break
        splits.append({"train_start": train_start, "train_end": min(train_end, end),
                        "test_start": test_start, "test_end": min(test_end, end)})
        train_start = train_start + pd.DateOffset(years=step_years)
        if train_start >= end:
            break
    return splits


def walk_forward_research(feature_frames: dict, horizon: int = 20,
                           train_years: float = 3.0, test_years: float = 1.0) -> pd.DataFrame:
    if not feature_frames:
        return pd.DataFrame()
    all_dates = pd.DatetimeIndex(sorted(set().union(*[set(f.index) for f in feature_frames.values()])))
    splits = walk_forward_splits(all_dates, train_years=train_years, test_years=test_years, step_years=test_years)
    rows = []
    for i, sp in enumerate(splits):
        windowed = {t: f[(f.index >= sp["test_start"]) & (f.index < sp["test_end"])]
                    for t, f in feature_frames.items()}
        windowed = {t: f for t, f in windowed.items() if len(f) > 30}
        if not windowed:
            continue
        ev = run_event_study_all_models(windowed)
        if ev.empty:
            continue
        for model in ev["Entry_Model"].unique():
            stats = performance_stats(ev[ev["Entry_Model"] == model][f"Fwd_Ret_{horizon}D"])
            stats.update({"fold": i, "test_start": sp["test_start"], "test_end": sp["test_end"], "entry_model": model})
            rows.append(stats)
    return pd.DataFrame(rows)


# --------------------------- MODE 3 EXECUTION ---------------------------
walk_forward_df = walk_forward_research(feature_frames, horizon=20, train_years=3.0, test_years=1.0)
print(f"=== MODE 3: WALK-FORWARD RESEARCH ({'SYNTHETIC DEMO' if IS_SYNTHETIC_DEMO else 'LIVE'}) ===")
if walk_forward_df.empty:
    print("Insufficient history for a walk-forward split (need several years of data per stock).")
else:
    print(walk_forward_df[["fold", "test_start", "test_end", "entry_model", "n_trades", "avg_return_pct", "win_rate_pct"]]
          .to_string(index=False))
    print("\nIf later folds show materially worse expectancy than earlier folds, treat the")
    print("in-sample Event Study (Cell 19) results with caution - this is expected, honest behavior")
    print("to check for, not a bug to explain away.")


## Cell 21 — Parameter sensitivity / robustness (Sections 29–30)

**Section 30 is explicit: 10/20/89/200 is the user's specification, NOT proven optimal by this notebook.** Each EMA role is perturbed one at a time around the base spec; the full grid distribution is reported so a genuine plateau is distinguishable from an overfit spike.

In [ ]:
def evaluate_param_grid(param_grid: dict, evaluate_fn, fixed_kwargs: dict = None) -> pd.DataFrame:
    fixed_kwargs = fixed_kwargs or {}
    keys = list(param_grid.keys())
    rows = []
    for combo in itertools.product(*[param_grid[k] for k in keys]):
        params = dict(zip(keys, combo))
        try:
            metrics = evaluate_fn(**params, **fixed_kwargs)
        except Exception as e:
            metrics = {"error": str(e)}
        rows.append({**params, **metrics})
    return pd.DataFrame(rows)


def plateau_summary(grid_df: pd.DataFrame, metric_col: str, param_cols: list) -> dict:
    if metric_col not in grid_df.columns or grid_df[metric_col].dropna().empty:
        return {"note": "no valid metric values in grid"}
    vals = grid_df[metric_col].dropna()
    best_row = grid_df.loc[vals.idxmax()]
    return {"grid_size": len(grid_df), "metric_mean": float(vals.mean()), "metric_median": float(vals.median()),
            "metric_std": float(vals.std()), "metric_min": float(vals.min()), "metric_max": float(vals.max()),
            "best_params": {c: best_row[c] for c in param_cols}, "best_metric": float(best_row[metric_col]),
            "spread_best_minus_median": float(vals.max() - vals.median())}


def ema_period_sensitivity(universe: dict, index_df: pd.DataFrame,
                            fast_grid=(8, 10, 12), medium_grid=(18, 20, 22),
                            structural_grid=(80, 89, 100), long_grid=(180, 200, 220),
                            horizon: int = 20, n_tickers_subset: int = None) -> pd.DataFrame:
    """Perturbs ONE EMA role at a time (holding the other three at the base
    spec) to keep the grid tractable, per Section 29's plateau-not-optimum guidance."""
    if n_tickers_subset is not None:
        universe = {k: universe[k] for k in list(universe.keys())[:n_tickers_subset]}
    base = {"fast": CFG.EMA_FAST, "medium": CFG.EMA_MEDIUM, "structural": CFG.EMA_STRUCTURAL, "long": CFG.EMA_LONG}

    def evaluate(fast, medium, structural, long_):
        cfg = Config()
        cfg.EMA_FAST, cfg.EMA_MEDIUM, cfg.EMA_STRUCTURAL, cfg.EMA_LONG = fast, medium, structural, long_
        regime_df = compute_market_regime(index_df)
        frames = {}
        for ticker, daily_df in universe.items():
            try:
                frames[ticker] = build_stock_feature_frame(daily_df, index_df["Close"], regime_df, cfg)
            except Exception:
                continue
        if not frames:
            return {"n_trades": 0, "avg_return_pct": np.nan}
        ev = run_event_study_all_models(frames)
        if ev.empty:
            return {"n_trades": 0, "avg_return_pct": np.nan}
        stats = performance_stats(ev[f"Fwd_Ret_{horizon}D"])
        return {"n_trades": stats.get("n_trades", 0), "avg_return_pct": stats.get("avg_return_pct", np.nan)}

    rows = []
    for f in fast_grid:
        rows.append({"perturbed_param": "fast", **evaluate(f, base["medium"], base["structural"], base["long"]),
                     "fast": f, "medium": base["medium"], "structural": base["structural"], "long": base["long"]})
    for m in medium_grid:
        rows.append({"perturbed_param": "medium", **evaluate(base["fast"], m, base["structural"], base["long"]),
                     "fast": base["fast"], "medium": m, "structural": base["structural"], "long": base["long"]})
    for s in structural_grid:
        rows.append({"perturbed_param": "structural", **evaluate(base["fast"], base["medium"], s, base["long"]),
                     "fast": base["fast"], "medium": base["medium"], "structural": s, "long": base["long"]})
    for l in long_grid:
        rows.append({"perturbed_param": "long", **evaluate(base["fast"], base["medium"], base["structural"], l),
                     "fast": base["fast"], "medium": base["medium"], "structural": base["structural"], "long": l})
    return pd.DataFrame(rows)


# --------------------------- SENSITIVITY EXECUTION ---------------------------
# Kept to a small subset of tickers + a light grid by default (runtime).
# Widen fast_grid/medium_grid/structural_grid/long_grid and n_tickers_subset
# for a fuller study once the notebook is confirmed working end-to-end.
sensitivity_df = ema_period_sensitivity(
    universe, index_df, fast_grid=(8, 10, 12), medium_grid=(18, 20, 22),
    structural_grid=(89,), long_grid=(200,), horizon=20,
    n_tickers_subset=min(15, len(universe)),
)
print(f"=== SENSITIVITY / ROBUSTNESS ({'SYNTHETIC DEMO' if IS_SYNTHETIC_DEMO else 'LIVE'}) ===")
print(sensitivity_df.to_string(index=False))
plateau = plateau_summary(sensitivity_df, "avg_return_pct", ["fast", "medium", "structural", "long"])
print("\nPlateau summary (do NOT just look at best_metric):")
print(json.dumps({k: (v if not isinstance(v, dict) else {kk: (vv.item() if hasattr(vv, "item") else vv) for kk, vv in v.items()})
                   for k, v in plateau.items()}, indent=2, default=str))
print("\nA small spread_best_minus_median relative to metric_std suggests a plateau;")
print("a huge gap between best_metric and metric_median suggests an overfit spike -")
print("in that case 10/20/89/200 should NOT be replaced by the 'best' grid cell.")


## Cell 22 — Validation tests: unit tests (Section 37) & look-ahead bias tests (Section 38)

All tests use **synthetic** series generated inline (never live data), so this cell is fully self-contained and reproducible. It reports a pass/fail table and raises if anything fails - it does not silently pass.

In [ ]:
def reference_ema_pandas_adjust_false(series: pd.Series, period: int) -> pd.Series:
    """Trusted reference: pandas .ewm(adjust=False), seeded identically to
    compute_ema's 'sma' seeding so this is an apples-to-apples check of the
    RECURSIVE STEP (not a re-validation of the seed choice itself)."""
    values = series.to_numpy(dtype=float)
    n, seed_idx = len(values), period - 1
    if n <= seed_idx:
        return pd.Series(np.full(n, np.nan), index=series.index)
    seed_val = np.nanmean(values[:period])
    tail = pd.Series(values[seed_idx:]); tail.iloc[0] = seed_val
    ema_tail = tail.ewm(span=period, adjust=False).mean()
    out = np.full(n, np.nan); out[seed_idx:] = ema_tail.to_numpy()
    return pd.Series(out, index=series.index)


def validate_ema_against_reference(period=20, n_obs=500, tol=1e-8, seed=42):
    rng = np.random.default_rng(seed)
    prices = pd.Series(100 + np.cumsum(rng.normal(0, 1, n_obs)))
    diff = (compute_ema(prices, period, seed_method="sma") - reference_ema_pandas_adjust_false(prices, period)).abs()
    max_diff = diff.max(skipna=True)
    return {"test": "ema_vs_reference", "max_abs_diff": float(max_diff) if pd.notna(max_diff) else None,
            "tolerance": tol, "passed": bool(max_diff < tol) if pd.notna(max_diff) else False}


def _test_ema_vs_reference():
    r = validate_ema_against_reference(); assert r["passed"], r


def _test_bullish_crossover_detection():
    a, b = pd.Series([1, 1, 2, 3, 4], dtype=float), pd.Series([2, 2, 2, 2, 2], dtype=float)
    assert bullish_crossover(a, b).tolist() == [False, False, False, True, False]


def _test_bearish_crossover_detection():
    a, b = pd.Series([3, 3, 2, 1, 0], dtype=float), pd.Series([2, 2, 2, 2, 2], dtype=float)
    assert bearish_crossover(a, b).tolist() == [False, False, False, True, False]


def _test_alignment_classification():
    state = classify_alignment(pd.Series([4, 1]), pd.Series([3, 2]), pd.Series([2, 3]), pd.Series([1, 4]))
    assert state.iloc[0] == STATE_BULLISH_ALIGNED and state.iloc[1] == STATE_BEARISH_ALIGNED


def _test_mixed_state():
    state = classify_alignment(pd.Series([2.0]), pd.Series([3.0]), pd.Series([1.0]), pd.Series([0.5]))
    assert state.iloc[0] == "MIXED"


def _test_cluster_width_nonnegative():
    rng = np.random.default_rng(0); dates = pd.bdate_range("2020-01-01", periods=400)
    df = pd.DataFrame({"Close": 100 + np.cumsum(rng.normal(0, 1, 400))}, index=dates)
    df = compute_cluster_metrics(compute_all_emas(df, suffix=""), suffix="")
    assert (df["Cluster_Width_Pct"].dropna() >= 0).all()


def _test_cluster_expansion_states_valid():
    rng = np.random.default_rng(1); dates = pd.bdate_range("2020-01-01", periods=400)
    df = pd.DataFrame({"Close": 100 + np.cumsum(rng.normal(0, 1, 400))}, index=dates)
    df = compute_cluster_expansion(compute_cluster_metrics(compute_all_emas(df, suffix=""), suffix=""), suffix="")
    assert set(df["Cluster_Expansion_State"].dropna().unique()).issubset({"EXPANDING", "CONTRACTING", "FLAT"})


def _test_swing_confirmation_lag():
    rng = np.random.default_rng(2); dates = pd.bdate_range("2021-01-01", periods=200)
    close = 100 + np.cumsum(rng.normal(0, 1, 200))
    df = pd.DataFrame({"Open": close, "High": close + 1, "Low": close - 1, "Close": close, "Volume": 1000}, index=dates)
    swing = detect_swing_points(df, left=5, right=5)
    swing_dates = swing.index[swing["Is_Swing_Low"]]
    assert len(swing_dates) > 0
    first = swing_dates[0]
    confirm_date = swing.loc[first, "Swing_Low_Confirmed_At"]
    day_before = swing.index[swing.index.get_loc(confirm_date) - 1]
    assert latest_confirmed_swing_low(swing, day_before)["date"] != first


def _test_weekly_no_lookahead():
    rng = np.random.default_rng(4); dates = pd.bdate_range("2020-01-01", periods=700)
    close = 100 + np.cumsum(rng.normal(0, 1, 700))
    daily = pd.DataFrame({"Open": close, "High": close + 1, "Low": close - 1, "Close": close, "Volume": 1000}, index=dates)
    cols = ["EMA10_W", "EMA20_W", "EMA89_W", "EMA200_W"]
    d_full = compute_all_emas(daily, suffix="_D")
    w_full = compute_all_emas(build_true_weekly_ohlc(daily), suffix="_W")
    merged_full = attach_last_known_weekly(d_full, w_full, cols)

    cutoff = dates[550]; daily_partial = daily.loc[:cutoff]
    d_partial = compute_all_emas(daily_partial, suffix="_D")
    w_partial = compute_all_emas(build_true_weekly_ohlc(daily_partial), suffix="_W")
    merged_partial = attach_last_known_weekly(d_partial, w_partial, cols)

    check_date = dates[500]
    diff = (merged_full.loc[check_date, cols].astype(float) - merged_partial.loc[check_date, cols].astype(float)).abs().max()
    assert diff < 1e-9, f"future data changed a past weekly value! diff={diff}"


def _test_swing_no_lookahead():
    rng = np.random.default_rng(5); dates = pd.bdate_range("2021-01-01", periods=300)
    close = 100 + np.cumsum(rng.normal(0, 1, 300))
    df = pd.DataFrame({"Open": close, "High": close + 1, "Low": close - 1, "Close": close, "Volume": 1000}, index=dates)
    swing = detect_swing_points(df, left=5, right=5)
    swing_lows = swing.index[swing["Is_Swing_Low"]]
    assert len(swing_lows) > 0
    for sw_date in swing_lows[:5]:
        confirm_date = swing.loc[sw_date, "Swing_Low_Confirmed_At"]
        loc = swing.index.get_loc(confirm_date)
        if loc == 0:
            continue
        before = swing.index[loc - 1]
        assert latest_confirmed_swing_low(swing, before)["date"] != sw_date, f"swing at {sw_date} leaked early"


ALL_TESTS = [
    ("EMA vs trusted reference implementation", _test_ema_vs_reference),
    ("Bullish crossover detection", _test_bullish_crossover_detection),
    ("Bearish crossover detection", _test_bearish_crossover_detection),
    ("Bullish/Bearish alignment classification", _test_alignment_classification),
    ("Mixed alignment classification", _test_mixed_state),
    ("Cluster width is non-negative", _test_cluster_width_nonnegative),
    ("Cluster expansion states are valid categories", _test_cluster_expansion_states_valid),
    ("Swing point confirmation lag enforced", _test_swing_confirmation_lag),
    ("[LOOK-AHEAD] Weekly EMA never leaks future data into the past", _test_weekly_no_lookahead),
    ("[LOOK-AHEAD] Swing low never usable before its confirmation date", _test_swing_no_lookahead),
]


def run_all_validation_tests(raise_on_failure: bool = True) -> pd.DataFrame:
    rows = []
    for name, fn in ALL_TESTS:
        try:
            fn(); rows.append({"test": name, "passed": True, "error": None})
        except Exception as e:
            rows.append({"test": name, "passed": False, "error": f"{type(e).__name__}: {e}"})
    results = pd.DataFrame(rows)
    if raise_on_failure and not results["passed"].all():
        raise AssertionError(f"Validation FAILED:\n{results[~results['passed']].to_string(index=False)}")
    return results


validation_results_df = run_all_validation_tests()
print("=== VALIDATION TEST SUITE ===")
print(validation_results_df.to_string(index=False))
print(f"\nAll {len(validation_results_df)} tests passed." if validation_results_df["passed"].all() else "SOME TESTS FAILED - see above.")


## Cell 23 — Excel export (Section 40)

**Percentage-format fix**: values are stored as **decimal fractions** (e.g. `0.0231`) with a `'0.00%'` number format, so Excel displays `2.31%` — never the `2.31 -> 231%` bug from formatting a percent-unit value with a percent number format.

In [ ]:
FRACTION_ALREADY_0_1 = ("_Score", "_Percentile")  # already 0-1 fractions; format as percent, no /100 needed
FRACTION_EXACT_NAMES = {"Research_Score_Heuristic"}


def _is_already_fraction_col(col: str) -> bool:
    return col.endswith(FRACTION_ALREADY_0_1) or col in FRACTION_EXACT_NAMES


def _is_pct_unit_col(col: str) -> bool:
    """Matches any column whose value is expressed IN PERCENT UNITS (e.g. 2.31
    meaning 2.31%) - these need /100 before a percent number format is applied.
    Deliberately broad (substring 'Pct', case-sensitive, e.g. '20D_Win_Rate_Pct',
    'Cluster_Width_Pct', 'RS_60D' is handled separately below) so newly added
    columns don't silently fall through the crack that caused the historical bug."""
    if _is_already_fraction_col(col):
        return False
    if "Pct" in col:
        return True
    if col.startswith("RS_") and col.endswith("D"):   # RS_20D, RS_60D, RS_120D
        return True
    if col == "EMA_Slope":
        return True
    if col.startswith("Fwd_Ret_") and (col.endswith("D") or col.endswith("D_Net")):
        return True
    return False


def prepare_percent_columns_as_decimal(df: pd.DataFrame):
    out = df.copy()
    percent_cols = set()
    for col in out.columns:
        if _is_pct_unit_col(col):
            out[col] = pd.to_numeric(out[col], errors="coerce") / 100.0
            percent_cols.add(col)
        elif _is_already_fraction_col(col):
            percent_cols.add(col)  # already a 0-1 fraction; format as percent, no /100
    return out, percent_cols


def write_dataframe_sheet(wb: Workbook, sheet_name: str, df: pd.DataFrame, freeze_panes: str = "A2", add_table: bool = True):
    ws = wb.create_sheet(sheet_name)
    if df is None or df.empty:
        ws.append(["No data available for this sheet."]); return ws
    df_conv, percent_cols = prepare_percent_columns_as_decimal(df)
    ws.append(list(df_conv.columns))
    for _, row in df_conv.iterrows():
        ws.append([None if pd.isna(v) else (v.item() if hasattr(v, "item") else v) for v in row])
    percent_col_idx = {i for i, c in enumerate(df_conv.columns, start=1) if c in percent_cols}
    for col_idx in percent_col_idx:
        for r in range(2, ws.max_row + 1):
            cell = ws.cell(row=r, column=col_idx)
            if isinstance(cell.value, (int, float)):
                cell.number_format = "0.00%"
    if freeze_panes:
        ws.freeze_panes = freeze_panes
    if add_table and ws.max_row > 1 and ws.max_column > 0:
        table_ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
        safe_name = "T_" + "".join(c for c in sheet_name if c.isalnum())[:25]
        table = Table(displayName=safe_name, ref=table_ref)
        table.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
        ws.add_table(table)
    for col_idx, col_name in enumerate(df_conv.columns, start=1):
        ws.column_dimensions[get_column_letter(col_idx)].width = max(12, min(28, len(str(col_name)) + 4))
    return ws


def export_research_workbook(output_path: str, sheets: dict):
    wb = Workbook(); wb.remove(wb.active)
    for name, content in sheets.items():
        if isinstance(content, pd.DataFrame):
            write_dataframe_sheet(wb, name, content)
        else:
            ws = wb.create_sheet(name)
            for line in content:
                ws.append([line])
            ws.column_dimensions["A"].width = 100
    wb.save(output_path)
    return output_path


parameters_df = pd.DataFrame([{"Parameter": k, "Value": str(v)} for k, v in vars(Config).items() if not k.startswith("_")])
universe_export_df = universe_meta_df.copy()
universe_export_df["Universe_Mode"] = UNIVERSE_MODE_LABEL

readme_lines = [
    "NIFTY 200 EMA CLUSTER RESEARCH WORKBOOK",
    f"Generated: {dt.datetime.now().isoformat()}",
    f"Universe mode: {UNIVERSE_MODE_LABEL}",
    "",
    "Percentages are stored as DECIMAL FRACTIONS with a 0.00% cell format",
    "(e.g. 0.0231 displays as 2.31%) - this avoids the 2.31->231% bug.",
    "",
    "Sheets:",
    "  Live_Signals     - Mode 1 output, one row per ticker, latest date",
    "  Diagnostics      - full pass/fail/state audit trail per ticker",
    "  State_Log        - every Signal_State transition, all tickers",
    "  Universe         - constituent list actually used + universe mode label",
    "  Research_Summary - Section 47 comparison table across entry models",
    "  Event_Study       - every historical event with forward returns/MFE/MAE",
    "  Forward_Returns  - regime breakdown of forward returns (Section 48)",
    "  Sensitivity      - EMA-period sensitivity grid (Sections 29-30)",
    "  Parameters       - full Config dump (Section 44: no hidden magic numbers)",
    "  Scan_Log         - per-ticker errors encountered during this run",
    "",
    "No profitability claim is implied by any sheet in this workbook.",
]

OUTPUT_XLSX_PATH = "Nifty200_EMA_Cluster_Research.xlsx"
export_research_workbook(OUTPUT_XLSX_PATH, {
    "Live_Signals": scan_df,
    "Diagnostics": diagnostics_df,
    "State_Log": state_log_df,
    "Universe": universe_export_df,
    "Research_Summary": comparison_df,
    "Event_Study": event_df.drop(columns=["Entry_Model_Label"], errors="ignore"),
    "Forward_Returns": regime_breakdown_df,
    "Sensitivity": sensitivity_df,
    "Parameters": parameters_df,
    "Scan_Log": error_log_df,
    "README": readme_lines,
})
print(f"Workbook written to: {OUTPUT_XLSX_PATH}")

try:
    from google.colab import files
    files.download(OUTPUT_XLSX_PATH)
except Exception:
    print("(Not running in Colab, or google.colab.files unavailable - file is saved locally.)")


## Cell 24 — Final report: scanner summary, research report, Principal Engineer review (Sections 52–54)

In [ ]:
# --- Section 52: concise final daily scanner summary ---
print("=========================================")
print("NIFTY 200 EMA CLUSTER SCAN")
print("=========================================")
print(f"Date:            {pd.Timestamp.today().date()}")
print(f"Universe:        {UNIVERSE_MODE_LABEL}")
print(f"Stocks scanned:  {len(feature_frames)}")
print(f"Data failures:   {len(error_log_df)}")
if not scan_df.empty:
    print(f"Bullish cluster: {(scan_df['Daily_State'] == STATE_BULLISH_ALIGNED).sum()}")
    print(f"Fresh cluster transitions today: {scan_df['Entry_Model'].str.contains('EntryA').sum()}")
    print(f"Pullback setups: {(scan_df['Signal_State'] == 'PULLBACK').sum()}")
    print(f"Confirmed entries today: {(scan_df['Entry_Model'] != 'NONE').sum()}")
    print("\nTop candidates by heuristic research score (NOT a guaranteed trade list):")
    top = scan_df.sort_values("Research_Score_Heuristic", ascending=False).head(10)
    print(top[["Stock", "Daily_State", "Weekly_State", "Entry_Model", "Research_Score_Heuristic"]].to_string(index=False))

# --- Section 54: research report ---
def build_final_report(feature_frames, comparison_df, concentration_df, is_synthetic_demo) -> str:
    n_tickers = len(feature_frames)
    n_signals_total = int(concentration_df["n_signals"].sum()) if not concentration_df.empty else 0
    unique_tk = concentration_df["Ticker"].nunique() if not concentration_df.empty else 0
    lines = ["EMA CLUSTER STRATEGY - RESEARCH REPORT", "=" * 50]
    if is_synthetic_demo:
        lines += [f"*** DATA SOURCE: {SYNTHETIC_DEMO_TAG} - PLUMBING DEMONSTRATION ONLY ***",
                  "No conclusion below should be treated as a real market finding."]
    lines += ["", "1. What exactly is the tested EMA Cluster strategy?",
              "   EMA10/20/89/200 trend structure on Daily+Weekly timeframes, tested via",
              "   five independently-labeled entry definitions and four cluster-crossover",
              "   definitions (A-D), none presumed correct by construction.", "",
              "2. Which entry definitions were tested?"]
    lines += [f"   - {label}" for label in ENTRY_MODEL_COLS.values()]
    lines += ["", "3. Which filters were tested?",
              "   - EMA200 structural filter (price>EMA200, EMA89>EMA200, EMA200 rising)",
              "   - Market regime (Bull/Neutral/Bear) segmentation",
              "   - Relative strength (20/60/120D) vs index",
              "   - ATR-normalized pullback depth", ""]
    if not comparison_df.empty:
        best_row = comparison_df.sort_values(comparison_df.columns[3], ascending=False).iloc[0]
        lines += ["4. Which results were strongest (by median forward return, gross)?",
                  f"   - {best_row['Entry_Definition']} (n={best_row['N_Occurrences']})"]
    else:
        lines += ["4. No event-study results were available to rank (insufficient signals)."]
    lines += ["", "5. Are results consistent across regimes? See Forward_Returns/regime_breakdown_df -",
              "   do not assume yes; inspect it per Section 48's mandatory breakdown.", "",
              f"6. Observations: {n_tickers} tickers processed; {n_signals_total} total entry",
              f"   signals, spread across {unique_tk} unique tickers. If a small number of tickers",
              "   dominate the signal count, treat any strong aggregate statistic with caution.", "",
              "7. Largest weaknesses / what needs further validation:",
              "   - Walk-Forward (Cell 20) out-of-sample results should be reviewed before treating",
              "     any in-sample statistic (Cell 19) as predictive.",
              "   - The Sensitivity grid (Cell 21) should show a broad plateau, not a spike, before",
              "     10/20/89/200 is treated as anything more than the user's original specification.",
              "   - If run on the CURRENT NIFTY 200 list historically, results carry survivorship",
              "     bias unless historical constituent membership was used (see Universe sheet).",
              "   - Transaction-cost assumptions in Config are placeholders and must be confirmed",
              "     against the user's actual broker/exchange fee schedule.", "",
              "No profitability claim is made. All statistics are descriptive of the historical",
              "sample tested and are not a guarantee of future performance."]
    return "\n".join(lines)


print("\n" + build_final_report(feature_frames, comparison_df, concentration_df, IS_SYNTHETIC_DEMO))

# --- Section 53: Principal Engineer review ---
print("""
PRINCIPAL ENGINEER REVIEW (Section 53)
=======================================
- Look-ahead bias: weekly EMAs are attached via a strict '< as_of_date'
  as-of join; swings are only usable from their Confirmed_At date onward.
  Both are covered by automated tests in Cell 22.
- Survivorship bias: every run is labeled HISTORICAL-MEMBERSHIP-BACKTEST or
  CURRENT-UNIVERSE HISTORICAL SIMULATION (Universe sheet) - the latter is
  NOT presented as an unbiased historical NIFTY-200 backtest.
- EMA initialization: explicit SMA-seed recursion, cross-checked against a
  pandas .ewm(adjust=False) reference seeded identically (Cell 22).
- Crossover detection: explicit prev/current comparison, unit-tested.
- Stale data / partial universe: data-quality checks flag stale bars,
  duplicates, non-monotonic dates, non-positive prices; universe fetch
  fails the WHOLE scan rather than silently using a partial/stale list.
- Execution assumption: signal detection (EOD close) is explicitly separate
  from execution (configurable next-open/next-close/same-close; default
  next-open) - never assumed silently.
- Transaction costs: included and separated into gross vs net everywhere in
  the event study; cost values are placeholders the user must confirm.
- Parameter overfitting: the Sensitivity cell reports the FULL grid
  distribution alongside the single best cell; 10/20/89/200 is never
  asserted optimal.
- Hard-coded thresholds: cluster compression/expansion use ROLLING
  PERCENTILE thresholds (Config-parameterized), not fixed universal numbers.
- Arbitrary scoring: the composite research score is explicitly labeled a
  heuristic with configurable, visible weights; components are also
  reported individually.

Where the handwritten note is genuinely ambiguous (the exact cluster-
crossover trigger, and how many sessions apart the individual EMA
crossovers may occur under Definition A), this notebook exposes multiple
explicit, independently-testable interpretations (Definitions A-D, Entry
Models A-E) rather than silently picking one.
""")
